# Provenance-governed асистент пам'яті — датасет, симуляція та статистика

Програмна реалізація концептуальної рамки зі статті **D. Prokopovych-Tkachenko, «Provenance-governed agentic AI for personal knowledge and memory support: a conceptual framework»**.

Що робить цей ноутбук:

1. Встановлює пакет `pgmem` (код вбудовано нижче — нічого завантажувати не треба).
2. Генерує **синтетичний** датасет людей з порушеннями пам'яті (MCI, легка деменція, набуте ураження мозку) з повними **метаданими** (Frictionless Data Package: типи, одиниці, обмеження, походження).
3. Рахує описову статистику когорти за метаданими (вік, MoCA, PRMQ, CDR, умови проживання, цифрова грамотність…).
4. Відтворює сценарії статті (Олена, 72 роки) на реальному коді: рівняння (1)–(5), інваріанти I1–I4.
5. Порівнює три умови на однакових історіях — **нагадування**, **один агент + RAG**, **запропонована рамка** — за показниками TSR, FPR, F_ext, C (рівняння 6–7) та додатковими.
6. Перевіряє зв'язок метаданих з результатами (Kruskal–Wallis, Spearman, Mann–Whitney, логістична регресія) з кластерним бутстрепом.

> ⚠️ **Усі дані синтетичні.** Жоден запис не відповідає реальній людині; розподіли — проєктні припущення, а не клінічні оцінки. Результати показують, як поводяться правила рамки, і **не** доводять ефективності для людей (див. розділ 7.3 статті).


## 0. Середовище

In [1]:
%pip -q install pandas numpy scipy matplotlib tabulate
import os, sys
PROJECT = "pgmem_project"
os.makedirs(f"{PROJECT}/pgmem", exist_ok=True)
os.makedirs(f"{PROJECT}/data", exist_ok=True)
sys.path.insert(0, PROJECT)

Note: you may need to restart the kernel to use updated packages.


### 0.1 Код пакета `pgmem`

Наступні клітинки записують модулі пакета у `pgmem_project/pgmem/`. Їх можна згорнути.
*Альтернатива:* завантажте `pgmem_project.zip` з комплекту та розпакуйте (клітинка після модулів).

In [2]:
%%writefile pgmem_project/pgmem/__init__.py
"""pgmem — provenance-governed agentic memory support.

Reference implementation of the conceptual framework in
D. Prokopovych-Tkachenko, "Provenance-governed agentic AI for personal
knowledge and memory support: a conceptual framework".
"""
from .assistance import adapt_start_level, gate, next_level
from .governance import ActionQueue, PolicyEngine, QueuedAction
from .model import Episode, EpisodeState, EvidenceStatus, PersonalEventBase, Record
from .recall import Answer, Claim, GroundedRecall, NaiveRAG, Question, ReminderOnly, release, support_score

__version__ = "1.0.0"
__all__ = [
    "ActionQueue", "Answer", "Claim", "Episode", "EpisodeState", "EvidenceStatus", "GroundedRecall",
    "NaiveRAG", "PersonalEventBase", "PolicyEngine", "Question", "QueuedAction", "Record", "ReminderOnly",
    "adapt_start_level", "gate", "next_level", "release", "support_score",
]


Writing pgmem_project/pgmem/__init__.py


In [3]:
%%writefile pgmem_project/pgmem/model.py
"""Evidence-typed personal event base (Section 4.2, Equation 1).

Each record is r_i = (c_i, t_i, s_i, kappa_i, sigma_i, v_i, e_i):
content, time, source/provenance, confidence indicator, consent scope,
version and evidence status.  Corrections create new linked versions,
disagreements are marked as disputed, deletions leave only a minimal
audit event (no content).
"""
from __future__ import annotations

import itertools
from dataclasses import dataclass, field, replace
from datetime import datetime
from enum import Enum
from typing import Callable, Iterable


class EvidenceStatus(str, Enum):
    PLAN = "plan"                # entered as an intention / schedule
    SELF_REPORT = "self_report"  # the person said it happened
    OBSERVATION = "observation"  # sensor / external log observed it
    INFERENCE = "inference"      # drawn by the assistant
    DISPUTED = "disputed"        # incompatible records exist


# Which record statuses can support a claim that asserts a given status (I2).
# A plan supports "it was scheduled", never "it happened".
SUPPORTS: dict[EvidenceStatus, set[EvidenceStatus]] = {
    EvidenceStatus.PLAN: {EvidenceStatus.PLAN, EvidenceStatus.SELF_REPORT, EvidenceStatus.OBSERVATION},
    EvidenceStatus.SELF_REPORT: {EvidenceStatus.SELF_REPORT},
    EvidenceStatus.OBSERVATION: {EvidenceStatus.OBSERVATION},
    EvidenceStatus.INFERENCE: {EvidenceStatus.INFERENCE},
}

STATUS_STRENGTH = {
    EvidenceStatus.INFERENCE: 0,
    EvidenceStatus.PLAN: 1,
    EvidenceStatus.SELF_REPORT: 2,
    EvidenceStatus.OBSERVATION: 3,
}


@dataclass(frozen=True)
class Record:
    """One versioned entry of the personal event base."""

    record_id: str
    person_id: str
    subject: str                 # event identity, e.g. "appt:dr_kovalenko:2026-10-02"
    content: str                 # c_i
    time: datetime               # t_i
    source: str                  # s_i: calendar, user_voice, sensor:pillbox, carer_entry ...
    kappa: float                 # confidence indicator in [0, 1], not a probability
    scope: frozenset[str]        # sigma_i: recipients allowed to see it
    version: int = 1             # v_i
    status: EvidenceStatus = EvidenceStatus.PLAN  # e_i
    category: str = "general"    # data category used by consent scopes
    kind: str = "event"          # claim kind this record speaks to: scheduled/occurred/content
    value: str = ""              # normalised value of the claim (date, "true", note text)
    supersedes: str | None = None
    active: bool = True
    disputed: bool = False

    def __post_init__(self) -> None:
        if not 0.0 <= self.kappa <= 1.0:
            raise ValueError("kappa must lie in [0, 1]")


@dataclass
class AuditEvent:
    time: datetime
    actor: str
    action: str
    record_id: str
    detail: str = ""


class PersonalEventBase:
    """E = {r_1, ..., r_n} with versioning, disputes, deletion and audit."""

    def __init__(self, person_id: str):
        self.person_id = person_id
        self._records: dict[str, Record] = {}
        self.audit: list[AuditEvent] = []
        self._ids = itertools.count(1)
        self._listeners: list[Callable[[str], None]] = []

    # -- change notification (used by I4 cache invalidation) --------------
    def on_change(self, fn: Callable[[str], None]) -> None:
        self._listeners.append(fn)

    def _notify(self, subject: str) -> None:
        for fn in self._listeners:
            fn(subject)

    def new_id(self) -> str:
        return f"{self.person_id}-r{next(self._ids):05d}"

    # -- writes ------------------------------------------------------------
    def add(self, record: Record, actor: str = "system", when: datetime | None = None) -> Record:
        if record.person_id != self.person_id:
            raise ValueError("record belongs to another person")
        self._records[record.record_id] = record
        self.audit.append(AuditEvent(when or record.time, actor, "add", record.record_id, record.status.value))
        self._notify(record.subject)
        return record

    def correct(self, record_id: str, actor: str, when: datetime, **changes) -> Record:
        """Corrections never overwrite: a new linked version is created."""
        old = self._records[record_id]
        new = replace(old, record_id=self.new_id(), version=old.version + 1,
                      supersedes=old.record_id, active=True, **changes)
        self._records[old.record_id] = replace(old, active=False)
        self._records[new.record_id] = new
        self.audit.append(AuditEvent(when, actor, "correct", new.record_id, f"supersedes {old.record_id}"))
        self._notify(old.subject)
        return new

    def mark_disputed(self, record_ids: Iterable[str], actor: str, when: datetime) -> None:
        subjects = set()
        for rid in record_ids:
            r = self._records[rid]
            self._records[rid] = replace(r, disputed=True)
            subjects.add(r.subject)
            self.audit.append(AuditEvent(when, actor, "dispute", rid))
        for s in subjects:
            self._notify(s)

    def delete(self, record_id: str, actor: str, when: datetime) -> None:
        """Content is removed; the audit keeps only that a deletion happened."""
        r = self._records.pop(record_id)
        self.audit.append(AuditEvent(when, actor, "delete", record_id))
        self._notify(r.subject)

    # -- reads -------------------------------------------------------------
    def get(self, record_id: str) -> Record:
        return self._records[record_id]

    def all(self, include_inactive: bool = False) -> list[Record]:
        return [r for r in self._records.values() if include_inactive or r.active]

    def by_subject(self, subject: str, include_inactive: bool = False) -> list[Record]:
        return [r for r in self.all(include_inactive) if r.subject == subject]

    def __len__(self) -> int:
        return len(self._records)


class EpisodeState(str, Enum):
    """Explicit task-episode states (DP3, I3)."""

    PENDING = "pending"
    ACTIVE = "active"
    COMPLETED = "completed"   # always paired with a stated source
    POSTPONED = "postponed"
    DECLINED = "declined"
    UNKNOWN = "unknown"
    # Only produced by baselines that violate I3:
    MISSED_INFERRED = "missed_inferred"


@dataclass
class Episode:
    episode_id: str
    person_id: str
    task_type: str
    scheduled: datetime
    safety_critical: bool
    state: EpisodeState = EpisodeState.PENDING
    completion_source: str | None = None
    levels: list[int] = field(default_factory=list)

    def complete(self, source: str) -> None:
        if not source:
            raise ValueError("completion needs an explicit evidential basis (I3)")
        self.state = EpisodeState.COMPLETED
        self.completion_source = source


Writing pgmem_project/pgmem/model.py


In [4]:
%%writefile pgmem_project/pgmem/governance.py
"""Consent, access control and the dispatch-time permission check (I1, I4)."""
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime

from .model import AuditEvent


@dataclass
class PolicyEngine:
    """Revisable consent scopes keyed by (recipient, data category, action)."""

    person_id: str
    grants: set[tuple[str, str, str]] = field(default_factory=set)
    audit: list[AuditEvent] = field(default_factory=list)

    def grant(self, recipient: str, category: str, action: str) -> None:
        self.grants.add((recipient, category, action))

    def revoke(self, recipient: str, category: str, action: str) -> None:
        self.grants.discard((recipient, category, action))

    def allows(self, recipient: str, category: str, action: str) -> bool:
        # The person always sees and controls their own data.
        if recipient == self.person_id:
            return True
        return (recipient, category, action) in self.grants or (recipient, "*", action) in self.grants


@dataclass
class QueuedAction:
    key: str            # idempotency key: replay never duplicates (I4)
    kind: str           # message / disclosure / prompt
    recipient: str
    category: str
    payload: str
    created: datetime


class ActionQueue:
    """Actions are permitted at execution time, not at queueing time (I1)."""

    def __init__(self, policy: PolicyEngine):
        self.policy = policy
        self.pending: list[QueuedAction] = []
        self.sent: dict[str, QueuedAction] = {}
        self.cancelled: list[QueuedAction] = []

    def enqueue(self, action: QueuedAction) -> None:
        if action.key in self.sent or any(a.key == action.key for a in self.pending):
            return
        self.pending.append(action)

    def dispatch(self, now: datetime) -> list[QueuedAction]:
        delivered = []
        for a in self.pending:
            if a.key in self.sent:
                continue
            if self.policy.allows(a.recipient, a.category, a.kind):
                self.sent[a.key] = a
                delivered.append(a)
                self.policy.audit.append(AuditEvent(now, "assistant", f"send:{a.kind}", a.key, a.recipient))
            else:
                self.cancelled.append(a)
                self.policy.audit.append(AuditEvent(now, "assistant", "cancel", a.key, "permission changed"))
        self.pending = []
        return delivered


Writing pgmem_project/pgmem/governance.py


In [5]:
%%writefile pgmem_project/pgmem/assistance.py
"""Graduated assistance (Section 4.3, Equations 2 and 3).

Levels: L0 silent context, L1 brief reminder, L2 contextual cue,
L3 step-by-step guidance, L4 trusted-person involvement.
"""
from __future__ import annotations

LEVEL_NAMES = {
    0: "L0 контекст без підказки",
    1: "L1 коротке нагадування",
    2: "L2 контекстна підказка",
    3: "L3 покрокова підтримка",
    4: "L4 залучення довіреної особи",
}


def gate(permission_ok: bool, postponed: bool, declined: bool, progress_supports_escalation: bool) -> int:
    """Binary gate G_k: 1 only if permission allows another prompt, the user has
    neither postponed nor declined, and the progress check supports escalation.
    Missing/ambiguous observations must yield progress_supports_escalation=False."""
    return int(permission_ok and not postponed and not declined and progress_supports_escalation)


def next_level(level: int, g: int, l_max: int) -> int:
    """Equation (2): L_{k+1} = G_k * min(L_k + 1, L_max) + (1 - G_k) * L_k."""
    return g * min(level + 1, l_max) + (1 - g) * level


def adapt_start_level(start: int, consecutive_without_escalation: int, m: int,
                      l_min: int, user_agrees: bool) -> int:
    """Equation (3): L0(j+1) = max(L0(j) - 1, L_min) after m consecutive
    episodes completed without escalation, and only with the user's agreement."""
    if consecutive_without_escalation >= m and user_agrees:
        return max(start - 1, l_min)
    return start


Writing pgmem_project/pgmem/assistance.py


In [6]:
%%writefile pgmem_project/pgmem/recall.py
"""Grounded recall and uncertainty (Section 4.4, Equations 4 and 5).

Three assistants share one interface ``answer(question) -> Answer``:

* ``GroundedRecall``  – the proposed policy (permission filter, evidence-type
  check, conflict/identity check before thresholding, release rule).
* ``NaiveRAG``        – a single-agent retrieval assistant used as a baseline.
* ``ReminderOnly``    – a reminder interface that cannot answer questions.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable

import numpy as np

from .governance import PolicyEngine
from .model import STATUS_STRENGTH, SUPPORTS, EvidenceStatus, PersonalEventBase, Record

THETA_LOW = 0.40
THETA_HIGH = 0.75


@dataclass(frozen=True)
class Question:
    question_id: str
    person_id: str
    requester: str            # who is asking (the person or a trusted person)
    subject: str              # event identity the question refers to
    kind: str                 # scheduled / occurred / content
    category: str
    text: str = ""


@dataclass(frozen=True)
class Claim:
    """An atomic claim a_j (Min et al., 2023)."""

    subject: str
    kind: str
    value: str
    asserted_status: EvidenceStatus
    record_id: str | None     # None => no record behind the claim (fabrication)
    record_version: int | None = None


@dataclass
class Answer:
    decision: str             # answer / answer_uncertain / abstain / clarify / refuse / none
    claims: list[Claim] = field(default_factory=list)
    s_min: float | None = None
    text: str = ""
    disclosed_records: list[str] = field(default_factory=list)

    @property
    def substantive(self) -> bool:
        return self.decision in ("answer", "answer_uncertain")


def support_score(claim: Claim, records: list[Record],
                  sup: Callable[[Claim, Record], float]) -> float:
    """Equation (4): S(a_j) = max_{r in R(q)} sup(a_j, r) * kappa_r.
    Heuristic, neither a probability nor independent verification."""
    if not records:
        return 0.0
    return max(sup(claim, r) * r.kappa for r in records)


def release(s_min: float, theta_low: float = THETA_LOW, theta_high: float = THETA_HIGH) -> str:
    """Equation (5)."""
    if s_min >= theta_high:
        return "answer"
    if s_min >= theta_low:
        return "answer_uncertain"
    return "abstain"


def exact_sup(claim: Claim, r: Record) -> float:
    """Reference entailment: identity, kind, value and evidence type must match."""
    if r.subject != claim.subject or r.kind != claim.kind or r.value != claim.value:
        return 0.0
    return 1.0 if r.status in SUPPORTS[claim.asserted_status] else 0.0


def noisy_sup(rng: np.random.Generator, sd: float = 0.12) -> Callable[[Claim, Record], float]:
    """An imperfect entailment estimator, as any real NLI model would be."""
    def f(claim: Claim, r: Record) -> float:
        base = exact_sup(claim, r)
        return float(np.clip(base + rng.normal(0, sd) - (0.05 if base else 0.0), 0, 1))
    return f


_TEMPLATES = {
    ("scheduled", EvidenceStatus.PLAN): "У вашому календарі записано: {value}.",
    ("occurred", EvidenceStatus.SELF_REPORT): "Ви зафіксували {time}, що це зроблено. Незалежно я це не перевіряв.",
    ("occurred", EvidenceStatus.OBSERVATION): "Це зафіксовано джерелом «{source}» {time}.",
    ("content", EvidenceStatus.PLAN): "У нотатці записано: «{value}».",
    ("content", EvidenceStatus.SELF_REPORT): "Ви занотували: «{value}».",
}


class GroundedRecall:
    """The proposed recall policy (Figure 2b)."""

    def __init__(self, base: PersonalEventBase, policy: PolicyEngine,
                 sup: Callable[[Claim, Record], float] = exact_sup,
                 theta_low: float = THETA_LOW, theta_high: float = THETA_HIGH,
                 identity_confusion: float = 0.0, rng: np.random.Generator | None = None):
        self.base, self.policy, self.sup = base, policy, sup
        self.theta_low, self.theta_high = theta_low, theta_high
        self.identity_confusion = identity_confusion
        self.rng = rng or np.random.default_rng(0)
        self._cache: dict[tuple, tuple[Answer, tuple]] = {}
        base.on_change(self._invalidate)

    # I4: corrections, disputes and deletions invalidate cached answers.
    def _invalidate(self, subject: str) -> None:
        for key in [k for k in self._cache if k[1] == subject]:
            del self._cache[key]

    def _retrieve(self, q: Question) -> list[Record]:
        recs = [r for r in self.base.all()
                if self.policy.allows(q.requester, r.category, "read")]       # I1
        hits = [r for r in recs if r.subject == q.subject]
        if self.identity_confusion and self.rng.random() < self.identity_confusion:
            # Retriever slips in a record about a similar but different event.
            similar = [r for r in recs if r.subject != q.subject and r.subject.split(":")[0] == q.subject.split(":")[0]]
            if similar:
                hits = hits + [similar[int(self.rng.integers(len(similar)))]]
        return hits

    def answer(self, q: Question) -> Answer:
        # Permissions travel with cached answers: any grant change is a cache miss (I1, I4).
        key = (q.requester, q.subject, q.kind, frozenset(self.policy.grants))
        if key in self._cache:
            return self._cache[key][0]
        ans = self._answer(q)
        self._cache[key] = (ans, tuple(c.record_id for c in ans.claims))
        return ans

    def _answer(self, q: Question) -> Answer:
        if not self.policy.allows(q.requester, q.category, "read"):
            return Answer("refuse", text="Я не можу поділитися цією інформацією без вашого дозволу.")
        hits = self._retrieve(q)
        if not hits:
            return Answer("abstain", text="У мене немає жодного запису про це, і я не можу цього знати.")
        # Conflict / identity checks precede thresholding: a max cannot resolve contradictions.
        if any(r.disputed for r in hits):
            return Answer("clarify", text="Записи про це суперечать один одному. Давайте уточнимо разом.")
        if len({r.subject for r in hits}) > 1:
            return Answer("clarify", text="Я знайшов записи про кілька схожих подій. Яку саме ви маєте на увазі?")
        values = {(r.kind, r.value) for r in hits if r.kind == q.kind}
        if len(values) > 1:
            return Answer("clarify", text="Є кілька несумісних записів. Давайте уточнимо.")

        # Draft: the strongest claim the records allow, never stronger (I2).
        same_kind = [r for r in hits if r.kind == q.kind]
        limit_note = ""
        if q.kind == "occurred" and not same_kind:
            # Only a plan exists: answer about the schedule and state the limit.
            same_kind = [r for r in hits if r.kind == "scheduled"]
            limit_note = " Ці записи не підтверджують, що подія відбулася."
        if not same_kind:
            return Answer("abstain", text="Записи не стосуються саме цього питання.")
        best = max(same_kind, key=lambda r: (STATUS_STRENGTH.get(r.status, 0), r.time))
        claim = Claim(best.subject, best.kind, best.value, best.status, best.record_id, best.version)
        s = support_score(claim, hits, self.sup)
        decision = release(s, self.theta_low, self.theta_high)
        if decision == "abstain":
            return Answer("abstain", s_min=s, text="Я не впевнений і не хочу вигадувати відповідь.")
        tpl = _TEMPLATES.get((claim.kind, claim.asserted_status), "Запис: {value}.")
        text = tpl.format(value=best.value, source=best.source, time=best.time.strftime("%d.%m о %H:%M")) + limit_note
        if decision == "answer_uncertain":
            text += " (Підтримка запису неповна — варто перевірити.)"
        return Answer(decision, [claim], s, text, [best.record_id])


class NaiveRAG:
    """Single-agent retrieval assistant without evidence typing or governance."""

    def __init__(self, base: PersonalEventBase, rng: np.random.Generator,
                 p_fabricate: float = 0.4, p_stale: float = 0.3):
        self.base, self.rng = base, rng
        self.p_fabricate, self.p_stale = p_fabricate, p_stale

    def answer(self, q: Question) -> Answer:
        pool = self.base.all(include_inactive=self.rng.random() < self.p_stale)
        hits = [r for r in pool if r.subject == q.subject]          # no permission filter
        if not hits:
            if self.rng.random() < self.p_fabricate:
                fake = Claim(q.subject, q.kind, "true" if q.kind == "occurred" else "вигадано",
                             EvidenceStatus.OBSERVATION, None)
                return Answer("answer", [fake], 1.0, "Так, це було.")
            return Answer("abstain", text="Не знаю.")
        best = max(hits, key=lambda r: r.kappa)                       # highest score wins
        value = best.value
        if q.kind == "occurred" and best.kind == "scheduled":
            value = "true"                                            # plan read as attendance
        claim = Claim(q.subject, q.kind, value, EvidenceStatus.OBSERVATION, best.record_id, best.version)
        return Answer("answer", [claim], best.kappa, f"Так: {value}.", [best.record_id])


class ReminderOnly:
    def answer(self, q: Question) -> Answer:
        return Answer("none", text="(інтерфейс нагадувань не відповідає на запитання)")


Writing pgmem_project/pgmem/recall.py


In [7]:
%%writefile pgmem_project/pgmem/metrics.py
"""Evaluation indicators (Section 6, Equations 6 and 7)."""
from __future__ import annotations

import math


def _ratio(num: float, den: float) -> float:
    return num / den if den else math.nan


def tsr(n_completed: int, n_scheduled: int) -> float:
    """Task success rate TSR = N_c / N_s."""
    return _ratio(n_completed, n_scheduled)


def fpr(n_false_prompts: int, n_prompts: int) -> float:
    """False-prompt rate FPR = N_f / N_p."""
    return _ratio(n_false_prompts, n_prompts)


def f_ext(n_supported: int, n_released: int) -> float:
    """Externally assessed recall faithfulness F_ext = N_e / N_r.
    Undefined (NaN) when no claim is released — never reported as perfect."""
    return _ratio(n_supported, n_released)


def coverage(n_substantive: int, n_questions: int) -> float:
    """Answer coverage C = N_a / N_q."""
    return _ratio(n_substantive, n_questions)


def independence_index(n_completed_l0_l1: int, n_completed: int) -> float:
    """Share of completions achieved at L0-L1."""
    return _ratio(n_completed_l0_l1, n_completed)


def per_100(n_events: int, n_answers: int) -> float:
    """E.g. fabricated recollections per 100 substantive answers."""
    return 100.0 * _ratio(n_events, n_answers)


Writing pgmem_project/pgmem/metrics.py


In [8]:
%%writefile pgmem_project/pgmem/synth.py
"""Synthetic cohort and personal histories (Section 6, stage 1).

IMPORTANT: every value produced here is synthetic.  Distributions are
plausible design assumptions for people with memory impairment (MCI,
mild dementia, acquired brain injury), not empirical estimates, and no
generated person corresponds to a real individual.
"""
from __future__ import annotations

from datetime import datetime, timedelta

import numpy as np
import pandas as pd

START = datetime(2026, 9, 1)

DIAGNOSES = {
    # name: (share, age mean, age sd, age min, age max, MoCA mean, MoCA sd, CDR global)
    "MCI": (0.50, 73, 6.5, 55, 92, 22.5, 2.0, 0.5),
    "mild_dementia": (0.25, 78, 6.0, 60, 95, 17.5, 2.5, 1.0),
    "ABI": (0.25, 46, 14.0, 18, 80, 23.5, 3.0, np.nan),
}

TASKS = {
    # task: (every n days, safety critical, sensorable, hour, category)
    "medication_morning": (1, True, True, 8, "medication"),
    "daily_walk": (1, False, True, 11, "activity"),
    "appointment_prep": (7, False, False, 9, "appointments"),
    "bill_payment": (14, False, False, 15, "finance"),
    "call_family": (3, False, False, 18, "social"),
}

DOCTORS = ["Коваленко", "Шевчук", "Бондар", "Мельник", "Ткаченко", "Кравець"]
RELATIVES = ["Андрій", "Ірина", "Олег", "Марія", "Тарас", "Наталія"]
NOTES = ["наступний огляд через три місяці", "купити хліб і молоко", "подзвонити в ЖЕК",
         "пароль від Wi-Fi на холодильнику", "забрати окуляри з ремонту", "привітати сусідку з днем народження"]


def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def generate_cohort(n: int = 240, seed: int = 2026) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    names = list(DIAGNOSES)
    shares = np.array([DIAGNOSES[d][0] for d in names])
    dx = rng.choice(names, size=n, p=shares / shares.sum())
    rows = []
    for i, d in enumerate(dx):
        _, am, asd, amin, amax, mm, msd, cdr = DIAGNOSES[d]
        age = int(np.clip(round(rng.normal(am, asd)), amin, amax))
        moca = int(np.clip(round(rng.normal(mm, msd)), 8, 30))
        edu = int(np.clip(round(rng.normal(13, 3)), 6, 22))
        # MoCA adds one point for <= 12 years of education
        moca_adj = min(30, moca + (1 if edu <= 12 else 0))
        ability = (moca - 22) / 4 + rng.normal(0, 0.5)
        # PRMQ subscales: 8-40, higher = more frequent everyday failures
        prmq_p = int(np.clip(round(24 - 4.5 * ability + rng.normal(0, 3)), 8, 40))
        prmq_r = int(np.clip(round(22 - 4.0 * ability + rng.normal(0, 3)), 8, 40))
        older = age >= 65
        lives_alone = bool(rng.random() < (0.45 if older else 0.30))
        trusted = bool(rng.random() < 0.85)
        literacy = int(np.clip(round(rng.normal(3.4 - (0.6 if older else 0) + 0.3 * ability, 0.9)), 1, 5))
        rows.append(dict(
            participant_id=f"P{i + 1:03d}",
            age=age,
            age_band=pd.cut([age], [0, 49, 64, 74, 84, 120], labels=["<50", "50-64", "65-74", "75-84", "85+"])[0],
            sex=rng.choice(["F", "M"], p=[0.58, 0.42] if older else [0.40, 0.60]),
            diagnosis=d,
            cdr_global=cdr,
            moca=moca_adj,
            education_years=edu,
            prmq_prospective=prmq_p,
            prmq_retrospective=prmq_r,
            months_since_diagnosis=int(rng.gamma(2.0, 14.0)),
            lives_alone=lives_alone,
            has_trusted_person=trusted,
            trusted_relation=(rng.choice(["daughter", "son", "spouse", "sibling", "friend"]) if trusted else "none"),
            device=rng.choice(["smartphone", "tablet", "both"], p=[0.5, 0.2, 0.3]),
            digital_literacy=literacy,
            uses_voice=bool(rng.random() < 0.55),
            sensors_connected=bool(rng.random() < 0.6),
            call_log_connected=bool(rng.random() < 0.5),
            language=rng.choice(["uk", "en"], p=[0.8, 0.2]),
            agrees_to_level_reduction=bool(rng.random() < 0.7),
            ability_latent=round(float(ability), 3),
        ))
    return pd.DataFrame(rows)


def generate_episodes(cohort: pd.DataFrame, days: int = 28) -> pd.DataFrame:
    rows = []
    for p in cohort.itertuples():
        offset = int(p.participant_id[1:]) % 7
        for day in range(days):
            for task, (every, safety, sensorable, hour, cat) in TASKS.items():
                if (day + offset) % every:
                    continue
                rows.append(dict(
                    episode_id=f"{p.participant_id}-E{len(rows):06d}",
                    participant_id=p.participant_id,
                    day=day,
                    task_type=task,
                    category=cat,
                    safety_critical=safety,
                    sensorable=sensorable and p.sensors_connected,
                    scheduled=(START + timedelta(days=day, hours=hour)).isoformat(),
                ))
    return pd.DataFrame(rows)


def _rec(rows, pid, subject, content, t, source, kappa, scope, status, category, kind, value,
         version=1, supersedes="", active=True, disputed=False):
    rid = f"{pid}-r{sum(1 for r in rows if r['person_id'] == pid) + 1:05d}"
    rows.append(dict(record_id=rid, person_id=pid, subject=subject, content=content, time=t.isoformat(),
                     source=source, kappa=round(kappa, 3), scope=scope, version=version, status=status,
                     category=category, kind=kind, value=value, supersedes=supersedes,
                     active=active, disputed=disputed))
    return rid


def generate_memory_corpus(cohort: pd.DataFrame, seed: int = 7):
    """Personal event base records, recall questions and the independent
    reference history used to judge released claims (Section 6)."""
    rng = np.random.default_rng(seed)
    records, questions, reference = [], [], []

    def ref(pid, subject, kind, value):
        reference.append(dict(person_id=pid, subject=subject, kind=kind, value=value))

    def ask(pid, stratum, requester, subject, kind, category, ideal, text):
        questions.append(dict(question_id=f"{pid}-Q{sum(1 for q in questions if q['person_id'] == pid) + 1:03d}",
                              person_id=pid, stratum=stratum, requester=requester, subject=subject,
                              kind=kind, category=category, ideal=ideal, text=text))

    for p in cohort.itertuples():
        pid = p.participant_id
        trusted = f"trusted:{pid}"
        p_report = float(_sigmoid(0.3 + 0.8 * p.ability_latent))

        # 1-2. Past appointments: a plan; attendance only if self-reported.
        for j in range(3):
            doc = DOCTORS[(j + int(pid[1:])) % len(DOCTORS)]
            d = START + timedelta(days=int(rng.integers(0, 28)), hours=10)
            subj = f"appt:{doc}:{d.date()}"
            _rec(records, pid, subj, f"Прийом у лікаря {doc}", d, "calendar", 0.95,
                 "self", "plan", "appointments", "scheduled", str(d.date()))
            ref(pid, subj, "scheduled", str(d.date()))
            attended = rng.random() < 0.9
            ref(pid, subj, "occurred", str(attended).lower())
            reported = attended and rng.random() < p_report
            if reported:
                _rec(records, pid, subj, "Була на прийомі", d + timedelta(hours=3), "user_voice", 0.8,
                     "self", "self_report", "appointments", "occurred", "true")
            ref(pid, subj, "reported", "true" if reported else "none")
            ask(pid, "answerable_plan", pid, subj, "scheduled", "appointments", "answer",
                f"Коли в мене був запланований прийом у лікаря {doc}?")
            ask(pid, "attendance", pid, subj, "occurred", "appointments", "answer",
                f"Чи була я на прийомі в лікаря {doc}?")

        # 3. Medication: self-reports (which can themselves be wrong) or nothing.
        for j in range(4):
            d = START + timedelta(days=int(rng.integers(0, 28)), hours=8)
            subj = f"med:morning:{d.date()}"
            taken = rng.random() < 0.85
            reported = rng.random() < (p_report if taken else 0.05)
            ref(pid, subj, "occurred", str(taken).lower())
            ref(pid, subj, "reported", "true" if reported else "none")
            if reported:
                _rec(records, pid, subj, "Ранкові ліки прийнято", d + timedelta(minutes=47), "user_touch",
                     0.85, "self,trusted", "self_report", "medication", "occurred", "true")
            ask(pid, "self_report" if reported else "unanswerable", pid, subj, "occurred", "medication",
                "answer" if reported else "abstain", f"Чи прийняла я ранкові ліки {d.date()}?")

        # 4. Calls: answerable only when the call log is connected.
        for j in range(2):
            who = RELATIVES[(j + int(pid[1:])) % len(RELATIVES)]
            d = START + timedelta(days=int(rng.integers(0, 28)))
            subj = f"call:{who}:{d.date()}"
            called = rng.random() < 0.5
            ref(pid, subj, "occurred", str(called).lower())
            if p.call_log_connected and called:
                _rec(records, pid, subj, f"Вхідний дзвінок: {who}", d + timedelta(hours=19), "call_log",
                     0.98, "self", "observation", "social", "occurred", "true")
                stratum, ideal = "answerable_observation", "answer"
            else:
                stratum, ideal = "unanswerable", "abstain"
            ask(pid, stratum, pid, subj, "occurred", "social", ideal, f"Чи дзвонив мені {who} {d.date()}?")

        # 5. Contradiction: a self-report and a carer entry disagree.
        d = START + timedelta(days=int(rng.integers(0, 28)), hours=15)
        subj = f"bill:electricity:{d.date()}"
        ref(pid, subj, "occurred", str(rng.random() < 0.6).lower())
        _rec(records, pid, subj, "Рахунок за електроенергію сплачено", d, "user_voice", 0.8,
             "self", "self_report", "finance", "occurred", "true", disputed=True)
        _rec(records, pid, subj, "Рахунок не сплачено (запис доглядальника)", d + timedelta(hours=5),
             "carer_entry", 0.85, "self", "observation", "finance", "occurred", "false", disputed=True)
        ask(pid, "contradictory", pid, subj, "occurred", "finance", "clarify",
            "Чи сплатила я рахунок за електроенергію?")

        # 6. Stale: a rescheduled appointment (v1 superseded by v2).
        d1 = START + timedelta(days=int(rng.integers(0, 20)), hours=12)
        d2 = d1 + timedelta(days=int(rng.integers(2, 8)))
        subj = f"appt:стоматолог:{pid}"
        old = _rec(records, pid, subj, "Стоматолог", d1, "calendar", 0.95, "self", "plan",
                   "appointments", "scheduled", str(d1.date()), active=False)
        _rec(records, pid, subj, "Стоматолог (перенесено)", d2, "calendar", 0.95, "self", "plan",
             "appointments", "scheduled", str(d2.date()), version=2, supersedes=old)
        ref(pid, subj, "scheduled", str(d2.date()))
        ask(pid, "stale", pid, subj, "scheduled", "appointments", "answer", "Коли мій прийом у стоматолога?")

        # 7-8. Personal notes: the person may read them, a trusted person may not.
        for j in range(2):
            note = NOTES[(j * 3 + int(pid[1:])) % len(NOTES)]
            d = START + timedelta(days=int(rng.integers(0, 28)), hours=20)
            subj = f"note:{pid}:{j}"
            _rec(records, pid, subj, note, d, "user_voice", 0.9, "self", "self_report",
                 "personal_note", "content", note)
            ref(pid, subj, "content", note)
            ref(pid, subj, "reported", note)
            ask(pid, "answerable_note", pid, subj, "content", "personal_note", "answer", "Що я записала в нотатці?")
            ask(pid, "consent_restricted", trusted, subj, "content", "personal_note", "refuse",
                "(довірена особа) Що записано в її особистій нотатці?")

    return pd.DataFrame(records), pd.DataFrame(questions), pd.DataFrame(reference)


Writing pgmem_project/pgmem/synth.py


In [9]:
%%writefile pgmem_project/pgmem/simulate.py
"""Comparative simulation on identical synthetic histories (Section 6).

Conditions: ``reminder`` (fixed reminder interface), ``single_agent``
(retrieval assistant that escalates on timeouts and infers outcomes from
missing data) and ``proposed`` (the action-and-evidence contract).
Person behaviour uses common random numbers, so the three conditions
face exactly the same simulated people and days.
"""
from __future__ import annotations

from datetime import datetime

import numpy as np
import pandas as pd

from .assistance import adapt_start_level, gate, next_level
from .governance import PolicyEngine
from .model import STATUS_STRENGTH, SUPPORTS, EvidenceStatus, PersonalEventBase, Record
from .recall import GroundedRecall, NaiveRAG, Question, ReminderOnly, noisy_sup

CONDITIONS = ["reminder", "single_agent", "proposed"]
MAX_PROMPTS = 4
P_POSTPONE, P_DECLINE, P_DEVICE_DOWN, P_WITHDRAWN = 0.07, 0.03, 0.04, 0.08
M_ADAPT, L_MIN = 5, 0


def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def _respond(u: float, level: int, a: float) -> str:
    p_c = 0.8 if level >= 4 else float(_sigmoid(-1.4 + 0.7 * level + 0.9 * a))
    if u < p_c:
        return "complete"
    u -= p_c
    if u < P_POSTPONE:
        return "postpone"
    u -= P_POSTPONE
    if u < P_DECLINE:
        return "decline"
    rest = 1 - p_c - P_POSTPONE - P_DECLINE
    return "needs_help" if u < rest / 2 else "no_response"


def simulate_episodes(cohort: pd.DataFrame, episodes: pd.DataFrame, seed: int = 11) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    n = len(episodes)
    draws = dict(
        before=rng.random(n), known_before=rng.random(n), sr=rng.random(n), sensor=rng.random(n),
        withdrawn=rng.random(n), resp=rng.random((n, MAX_PROMPTS + 1)), dev=rng.random((n, MAX_PROMPTS + 1)),
    )
    people = cohort.set_index("participant_id")
    out = []
    for cond in CONDITIONS:
        start_level: dict[tuple[str, str], int] = {}
        streak: dict[tuple[str, str], int] = {}
        for i, ep in enumerate(episodes.itertuples()):
            p = people.loc[ep.participant_id]
            a = p.ability_latent
            p_sr = float(_sigmoid(0.3 + 0.8 * a))
            detected = bool(ep.sensorable and draws["sensor"][i] < 0.85)
            reports = draws["sr"][i] < p_sr
            withdrawn = draws["withdrawn"][i] < P_WITHDRAWN
            l_max = 4 if p.has_trusted_person else 3
            key = (ep.participant_id, ep.task_type)
            s0 = start_level.get(key, 1) if cond == "proposed" else 1

            done = draws["before"][i] < float(_sigmoid(-0.6 + 0.9 * a))
            completion_level = 0 if done else None
            known = None
            if done and detected and cond != "reminder":
                known = "sensor"
            elif done and draws["known_before"][i] < 0.4 * p_sr:
                known = "self_report"

            prompts = false_prompts = missed = escalations = 0
            unauthorised = cancelled = False
            level, max_level = s0, s0
            postponed = declined = resumed = ignored_decline = False
            k = 0
            while known is None and k < MAX_PROMPTS and not declined:
                if postponed:
                    if cond == "proposed":
                        if resumed:
                            break
                        resumed = True       # the user's chosen resumption rule: one later prompt
                        postponed = False
                    elif cond == "reminder":
                        postponed = False    # snooze -> next reminder is the resumption
                if level == 0:
                    # L0: context retained, no prompt; progress check before L1.
                    level = 1
                    max_level = max(max_level, 1)
                    continue
                if level == 4:
                    if cond == "proposed" and withdrawn:
                        cancelled = True     # I1: recheck at dispatch -> cancel
                        level = 3
                        l_max = 3
                        continue
                    if withdrawn:
                        unauthorised = True
                if draws["dev"][i, k] < P_DEVICE_DOWN:
                    missed += 1
                    k += 1
                    continue
                prompts += 1
                max_level = max(max_level, level)
                if done or postponed or ignored_decline:
                    false_prompts += 1
                k += 1
                if done:
                    if draws["resp"][i, k - 1] < p_sr:
                        known = "self_report"   # "Have you already...?" -> "yes"
                    continue
                r = _respond(draws["resp"][i, k - 1], level, a)
                if r == "complete":
                    done, completion_level = True, level
                    if reports:
                        known = "self_report"
                    elif detected and cond != "reminder":
                        known = "sensor"
                    continue
                if r == "postpone":
                    postponed = True
                    if cond == "single_agent":   # treats the delay as failure and escalates
                        new = next_level(level, 1, l_max)
                        escalations += new > level
                        level = new
                    continue
                if r == "decline":
                    declined = cond != "reminder"
                    ignored_decline = cond == "reminder"  # the reminder app keeps reminding
                    continue
                if cond == "reminder":
                    continue                  # fixed L1, no escalation
                if cond == "single_agent":
                    g = 1                     # timeout alone triggers escalation
                else:
                    g = gate(permission_ok=True, postponed=postponed, declined=declined,
                             progress_supports_escalation=(r == "needs_help"))
                new = next_level(level, g, l_max)
                escalations += new > level
                level = new

            # Recorded state of the episode
            if known:
                state, source = "completed", known
            elif declined:
                state, source = "declined", "user"
            elif postponed and cond == "proposed":
                state, source = "postponed", "user"
            elif cond == "proposed":
                state, source = "unknown", ""
            else:
                state, source = "missed_inferred", ""  # silence converted to failure (violates I3)

            if cond == "proposed":
                if state == "completed" and escalations == 0:
                    streak[key] = streak.get(key, 0) + 1
                else:
                    streak[key] = 0
                if not ep.safety_critical:
                    new_start = adapt_start_level(s0, streak[key], M_ADAPT, L_MIN, bool(p.agrees_to_level_reduction))
                    if new_start != s0:
                        streak[key] = 0
                    start_level[key] = new_start

            out.append(dict(
                condition=cond, episode_id=ep.episode_id, participant_id=ep.participant_id,
                task_type=ep.task_type, safety_critical=ep.safety_critical, day=ep.day,
                start_level=s0, max_level=max_level, prompts=prompts, false_prompts=false_prompts,
                missed_prompts=missed, escalations=escalations, completed_true=done,
                completion_level=completion_level, recorded_state=state, evidence_source=source,
                invented_outcome=state == "missed_inferred",
                wrong_failure=(state == "missed_inferred") and done,
                unauthorised_contact=unauthorised, contact_cancelled=cancelled,
            ))
    return pd.DataFrame(out)


# --------------------------------------------------------------------- recall
def _to_record(row) -> Record:
    return Record(
        record_id=row.record_id, person_id=row.person_id, subject=row.subject, content=row.content,
        time=datetime.fromisoformat(row.time), source=row.source, kappa=float(row.kappa),
        scope=frozenset(str(row.scope).split(",")), version=int(row.version),
        status=EvidenceStatus(row.status), category=row.category, kind=row.kind, value=str(row.value),
        supersedes=row.supersedes or None, active=bool(row.active), disputed=bool(row.disputed),
    )


def build_event_base(person_id: str, records: pd.DataFrame) -> PersonalEventBase:
    base = PersonalEventBase(person_id)
    for row in records[records.person_id == person_id].fillna({"supersedes": ""}).itertuples():
        base.add(_to_record(row), actor=row.source)
    return base


def build_policy(person_id: str) -> PolicyEngine:
    pol = PolicyEngine(person_id)
    trusted = f"trusted:{person_id}"
    pol.grant(trusted, "medication", "read")      # e.g. missed safety-critical tasks
    pol.grant(trusted, "medication", "message")
    return pol


def _ext_supported(claim, ref: dict) -> bool:
    if claim.kind == "occurred" and claim.asserted_status == EvidenceStatus.SELF_REPORT:
        return ref.get((claim.subject, "reported")) == claim.value
    return ref.get((claim.subject, claim.kind)) == claim.value


def simulate_recall(cohort, records, questions, reference, seed: int = 13) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    out = []
    for pid in cohort.participant_id:
        recs = records[records.person_id == pid]
        ref = {(r.subject, r.kind): str(r.value) for r in reference[reference.person_id == pid].itertuples()}
        base = build_event_base(pid, recs)
        policy = build_policy(pid)
        by_id = {r.record_id: r for r in base.all(include_inactive=True)}
        assistants = {
            "reminder": ReminderOnly(),
            "single_agent": NaiveRAG(base, np.random.default_rng(rng.integers(1 << 31))),
            "proposed": GroundedRecall(base, policy, sup=noisy_sup(np.random.default_rng(rng.integers(1 << 31))),
                                       identity_confusion=0.03, rng=np.random.default_rng(rng.integers(1 << 31))),
        }
        for q in questions[questions.person_id == pid].itertuples():
            question = Question(q.question_id, pid, q.requester, q.subject, q.kind, q.category, q.text)
            for cond, asst in assistants.items():
                ans = asst.answer(question)
                n_sup = n_fab = n_over = n_stale = n_wrong = 0
                for c in ans.claims:
                    if c.subject != q.subject:
                        n_wrong += 1          # accurate record, wrong question: unsupported
                    else:
                        n_sup += _ext_supported(c, ref)
                    if c.record_id is None:
                        n_fab += 1
                        continue
                    r = by_id[c.record_id]
                    n_over += r.status not in SUPPORTS.get(c.asserted_status, set()) or \
                        STATUS_STRENGTH.get(c.asserted_status, 0) > STATUS_STRENGTH.get(r.status, 0)
                    n_stale += not r.active
                disclosed = any(not policy.allows(q.requester, by_id[rid].category, "read")
                                for rid in ans.disclosed_records)
                decision_class = "answer" if ans.substantive else ans.decision
                out.append(dict(
                    condition=cond, question_id=q.question_id, participant_id=pid, stratum=q.stratum,
                    ideal=q.ideal, decision=ans.decision, substantive=ans.substantive,
                    appropriate=decision_class == q.ideal or (q.ideal == "refuse" and ans.decision == "abstain"),
                    claims_released=len(ans.claims), claims_supported=n_sup, fabricated=n_fab,
                    overstated=n_over, stale_used=n_stale, wrong_event=n_wrong, consent_violation=disclosed,
                    s_min=ans.s_min, answer_text=ans.text,
                ))
    return pd.DataFrame(out)


Writing pgmem_project/pgmem/simulate.py


In [10]:
%%writefile pgmem_project/pgmem/stats.py
"""Statistics over the cohort metadata and the simulated outcomes."""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats

from . import metrics as M


def describe_cohort(cohort: pd.DataFrame) -> pd.DataFrame:
    """Table of cohort characteristics by diagnosis (mean ± SD, %)."""
    def summarise(g: pd.DataFrame) -> pd.Series:
        ms = lambda c: f"{g[c].mean():.1f} ± {g[c].std():.1f}"
        pct = lambda c: f"{100 * g[c].mean():.0f}%"
        return pd.Series({
            "N": len(g),
            "Вік, роки": ms("age"),
            "Жінки": f"{100 * (g.sex == 'F').mean():.0f}%",
            "MoCA (0–30)": ms("moca"),
            "PRMQ проспективна (8–40)": ms("prmq_prospective"),
            "PRMQ ретроспективна (8–40)": ms("prmq_retrospective"),
            "Освіта, роки": ms("education_years"),
            "Місяців від діагнозу": ms("months_since_diagnosis"),
            "Живе сам(а)": pct("lives_alone"),
            "Є довірена особа": pct("has_trusted_person"),
            "Цифрова грамотність (1–5)": ms("digital_literacy"),
            "Підключені сенсори": pct("sensors_connected"),
            "Підключений журнал дзвінків": pct("call_log_connected"),
        })
    table = cohort.groupby("diagnosis").apply(summarise, include_groups=False).T
    table["Усього"] = summarise(cohort)
    return table


def memory_impairment_profile(cohort: pd.DataFrame) -> pd.DataFrame:
    """Severity bands from MoCA (common cut-offs: >=26 normal, 18–25 mild,
    10–17 moderate, <10 severe) crossed with diagnosis."""
    bands = pd.cut(cohort.moca, [-1, 9, 17, 25, 30], labels=["<10 тяжке", "10–17 помірне", "18–25 легке", "26–30 норма"])
    return pd.crosstab(cohort.diagnosis, bands, margins=True, margins_name="Усього")


def participant_metrics(ep: pd.DataFrame, rc: pd.DataFrame) -> pd.DataFrame:
    """Per participant x condition indicators (Eq. 6–7 and Section 6 extras)."""
    e = ep.assign(l01=ep.completed_true & (ep.completion_level.fillna(9) <= 1))
    eg = e.groupby(["participant_id", "condition"]).agg(
        n_scheduled=("episode_id", "size"), n_completed=("completed_true", "sum"),
        n_prompts=("prompts", "sum"), n_false=("false_prompts", "sum"), n_l01=("l01", "sum"),
        n_missed=("missed_prompts", "sum"), n_invented=("invented_outcome", "sum"),
        n_wrong_failure=("wrong_failure", "sum"), n_unauth=("unauthorised_contact", "sum"),
    )
    r = rc.assign(
        unnecessary_abstain=rc.decision.isin(["abstain", "refuse"]) & (rc.ideal == "answer"),
        appropriate_abstain=rc.decision.isin(["abstain", "refuse"]) & rc.ideal.isin(["abstain", "refuse"]),
        should_abstain=rc.ideal.isin(["abstain", "refuse"]),
    )
    rg = r.groupby(["participant_id", "condition"]).agg(
        n_questions=("question_id", "size"), n_substantive=("substantive", "sum"),
        n_released=("claims_released", "sum"), n_supported=("claims_supported", "sum"),
        n_fabricated=("fabricated", "sum"), n_overstated=("overstated", "sum"),
        n_stale=("stale_used", "sum"), n_consent=("consent_violation", "sum"),
        n_unnec_abstain=("unnecessary_abstain", "sum"), n_appr_abstain=("appropriate_abstain", "sum"),
        n_should_abstain=("should_abstain", "sum"), n_appropriate=("appropriate", "sum"),
    )
    pm = eg.join(rg).reset_index()
    return add_rates(pm)


def add_rates(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    v = np.vectorize
    df["TSR"] = v(M.tsr)(df.n_completed, df.n_scheduled)
    df["FPR"] = v(M.fpr)(df.n_false, df.n_prompts)
    df["independence"] = v(M.independence_index)(df.n_l01, df.n_completed)
    df["invented_outcome_rate"] = df.n_invented / df.n_scheduled
    df["F_ext"] = v(M.f_ext)(df.n_supported, df.n_released)
    df["coverage"] = v(M.coverage)(df.n_substantive, df.n_questions)
    df["fabricated_per_100"] = v(M.per_100)(df.n_fabricated, df.n_substantive)
    df["overstated_per_100"] = v(M.per_100)(df.n_overstated, df.n_substantive)
    df["appropriate_abstention"] = v(M._ratio)(df.n_appr_abstain, df.n_should_abstain)
    df["appropriate_behaviour"] = df.n_appropriate / df.n_questions
    return df


RATE_COLS = ["TSR", "FPR", "independence", "invented_outcome_rate", "F_ext", "coverage",
             "fabricated_per_100", "overstated_per_100", "appropriate_abstention", "appropriate_behaviour"]
COUNT_COLS = ["n_scheduled", "n_completed", "n_prompts", "n_false", "n_l01", "n_missed", "n_invented",
              "n_wrong_failure", "n_unauth", "n_questions", "n_substantive", "n_released", "n_supported",
              "n_fabricated", "n_overstated", "n_stale", "n_consent", "n_unnec_abstain", "n_appr_abstain",
              "n_should_abstain", "n_appropriate"]


def pooled(pm: pd.DataFrame, by: list[str]) -> pd.DataFrame:
    """Ratio-of-sums indicators for any grouping."""
    return add_rates(pm.groupby(by)[COUNT_COLS].sum()).reset_index()


def cluster_bootstrap(pm: pd.DataFrame, metric: str, n_boot: int = 1000, seed: int = 0) -> pd.DataFrame:
    """Pooled metric per condition with 95% CIs resampling participants (clusters)."""
    rng = np.random.default_rng(seed)
    pids = pm.participant_id.unique()
    idx = {p: g for p, g in pm.groupby("participant_id")}
    point = pooled(pm, ["condition"]).set_index("condition")[metric]
    boots = []
    for _ in range(n_boot):
        sample = pd.concat([idx[p] for p in rng.choice(pids, len(pids))])
        boots.append(pooled(sample, ["condition"]).set_index("condition")[metric])
    b = pd.DataFrame(boots)
    return pd.DataFrame({"estimate": point, "ci_low": b.quantile(0.025), "ci_high": b.quantile(0.975)})


def paired_differences(pm: pd.DataFrame, metrics: list[str], ref: str = "proposed",
                       n_boot: int = 2000, seed: int = 1) -> pd.DataFrame:
    """Per-participant paired differences (ref − baseline) with bootstrap CI
    and a Wilcoxon signed-rank test."""
    rng = np.random.default_rng(seed)
    wide = pm.pivot(index="participant_id", columns="condition", values=metrics)
    rows = []
    for m in metrics:
        for base in [c for c in wide[m].columns if c != ref]:
            d = (wide[m][ref] - wide[m][base]).dropna().to_numpy()
            if len(d) < 5:
                continue
            bs = [rng.choice(d, len(d)).mean() for _ in range(n_boot)]
            try:
                p = stats.wilcoxon(d).pvalue if np.any(d) else 1.0
            except ValueError:
                p = np.nan
            rows.append(dict(metric=m, comparison=f"{ref} − {base}", n=len(d), mean_diff=d.mean(),
                             ci_low=np.quantile(bs, 0.025), ci_high=np.quantile(bs, 0.975), wilcoxon_p=p))
    return pd.DataFrame(rows)


def group_tests(pm: pd.DataFrame, cohort: pd.DataFrame, condition: str = "proposed") -> pd.DataFrame:
    """Association between memory-related metadata and outcomes."""
    d = pm[pm.condition == condition].merge(cohort, on="participant_id")
    rows = []
    for metric in ["TSR", "independence", "FPR", "coverage"]:
        groups = [g[metric].dropna() for _, g in d.groupby("diagnosis")]
        h, p = stats.kruskal(*groups)
        rows.append(dict(test="Kruskal–Wallis за діагнозом", outcome=metric, statistic=h, p_value=p))
        for cov in ["moca", "prmq_prospective", "age", "digital_literacy"]:
            sub = d[[cov, metric]].dropna()
            rho, p = stats.spearmanr(sub[cov], sub[metric])
            rows.append(dict(test=f"Spearman ρ з {cov}", outcome=metric, statistic=rho, p_value=p))
        a, b = d[d.lives_alone][metric].dropna(), d[~d.lives_alone][metric].dropna()
        u, p = stats.mannwhitneyu(a, b)
        rows.append(dict(test="Mann–Whitney: живе сам(а) vs ні", outcome=metric, statistic=u, p_value=p))
    return pd.DataFrame(rows)


def logistic_completion(ep: pd.DataFrame, cohort: pd.DataFrame, condition: str = "proposed") -> pd.DataFrame:
    """Episode-level logistic regression of task completion on metadata
    (Newton–Raphson, no extra dependencies).  Odds ratios with Wald 95% CI;
    note that episodes are clustered within participants, so CIs are optimistic."""
    d = ep[ep.condition == condition].merge(cohort, on="participant_id")
    X = pd.DataFrame({
        "intercept": 1.0,
        "moca_per_point": d.moca - d.moca.mean(),
        "age_per_10y": (d.age - d.age.mean()) / 10,
        "lives_alone": d.lives_alone.astype(float),
        "sensors_connected": d.sensors_connected.astype(float),
        "digital_literacy": d.digital_literacy - d.digital_literacy.mean(),
        "safety_critical_task": d.safety_critical.astype(float),
    })
    y = d.completed_true.astype(float).to_numpy()
    Xv = X.to_numpy()
    beta = np.zeros(Xv.shape[1])
    for _ in range(50):
        mu = 1 / (1 + np.exp(-Xv @ beta))
        W = mu * (1 - mu)
        H = Xv.T @ (Xv * W[:, None])
        step = np.linalg.solve(H, Xv.T @ (y - mu))
        beta += step
        if np.abs(step).max() < 1e-8:
            break
    se = np.sqrt(np.diag(np.linalg.inv(H)))
    z = beta / se
    return pd.DataFrame({
        "term": X.columns, "OR": np.exp(beta), "ci_low": np.exp(beta - 1.96 * se),
        "ci_high": np.exp(beta + 1.96 * se), "p_value": 2 * stats.norm.sf(np.abs(z)),
    })


def recall_by_stratum(rc: pd.DataFrame) -> pd.DataFrame:
    g = rc.groupby(["stratum", "condition"]).agg(
        n=("question_id", "size"), coverage=("substantive", "mean"),
        appropriate=("appropriate", "mean"), fabricated=("fabricated", "sum"),
        overstated=("overstated", "sum"), consent_violations=("consent_violation", "sum"),
        stale=("stale_used", "sum"), wrong_event=("wrong_event", "sum"),
        released=("claims_released", "sum"), supported=("claims_supported", "sum"),
    )
    g["F_ext"] = g.supported / g.released.replace(0, np.nan)
    return g.reset_index()


Writing pgmem_project/pgmem/stats.py


In [11]:
%%writefile pgmem_project/pgmem/schema.py
"""Dataset metadata (Frictionless Data Package) and validation.

The same descriptor documents the synthetic data and lets a team check
that a real, ethically approved dataset with the same structure can be
analysed by this code without changes.
"""
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

F = lambda name, type_, desc, **c: {"name": name, "type": type_, "description": desc,
                                     **({"constraints": c} if c else {})}

RESOURCES = {
    "participants": ("Учасники когорти та метадані, пов'язані з пам'яттю (одна стрічка = одна людина)", [
        F("participant_id", "string", "Псевдонімізований ідентифікатор", required=True, unique=True, pattern=r"P\d{3}"),
        F("age", "integer", "Вік, роки", minimum=18, maximum=110),
        F("age_band", "string", "Вікова група", enum=["<50", "50-64", "65-74", "75-84", "85+"]),
        F("sex", "string", "Стать", enum=["F", "M"]),
        F("diagnosis", "string", "MCI — помірний когнітивний розлад; mild_dementia — легка деменція; ABI — набуте ураження мозку",
          enum=["MCI", "mild_dementia", "ABI"]),
        F("cdr_global", "number", "Clinical Dementia Rating, глобальна оцінка (порожньо для ABI)", minimum=0, maximum=3),
        F("moca", "integer", "Montreal Cognitive Assessment з поправкою на освіту (0–30)", minimum=0, maximum=30),
        F("education_years", "integer", "Роки освіти", minimum=0, maximum=30),
        F("prmq_prospective", "integer", "PRMQ, проспективна шкала (8–40, більше = частіші збої)", minimum=8, maximum=40),
        F("prmq_retrospective", "integer", "PRMQ, ретроспективна шкала (8–40)", minimum=8, maximum=40),
        F("months_since_diagnosis", "integer", "Місяців від встановлення діагнозу", minimum=0),
        F("lives_alone", "boolean", "Живе сам(а)"),
        F("has_trusted_person", "boolean", "Є погоджена довірена особа (R8)"),
        F("trusted_relation", "string", "Хто довірена особа"),
        F("device", "string", "Пристрій", enum=["smartphone", "tablet", "both"]),
        F("digital_literacy", "integer", "Самооцінка цифрової грамотності (1–5)", minimum=1, maximum=5),
        F("uses_voice", "boolean", "Користується голосовою взаємодією (R6)"),
        F("sensors_connected", "boolean", "Підключені сенсори (таблетниця, крокомір)"),
        F("call_log_connected", "boolean", "Підключено журнал дзвінків"),
        F("language", "string", "Мова інтерфейсу", enum=["uk", "en"]),
        F("agrees_to_level_reduction", "boolean", "Згода на зниження стартового рівня підтримки (рівн. 3)"),
        F("ability_latent", "number", "ЛИШЕ ДЛЯ СИМУЛЯЦІЇ: латентна здатність, з якої генеруються відповіді"),
    ]),
    "episodes": ("Заплановані епізоди завдань (рутини, R1–R4)", [
        F("episode_id", "string", "Ідентифікатор епізоду", required=True, unique=True),
        F("participant_id", "string", "Посилання на participants", required=True),
        F("day", "integer", "День спостереження (0–27)", minimum=0),
        F("task_type", "string", "Тип завдання",
          enum=["medication_morning", "daily_walk", "appointment_prep", "bill_payment", "call_family"]),
        F("category", "string", "Категорія даних для згоди"),
        F("safety_critical", "boolean", "Завдання, критичне для безпеки"),
        F("sensorable", "boolean", "Виконання може бути зафіксоване сенсором"),
        F("scheduled", "datetime", "Запланований час (ISO 8601)"),
    ]),
    "records": ("Персональна база подій: записи r_i = (c, t, s, κ, σ, v, e), рівняння (1)", [
        F("record_id", "string", "Ідентифікатор запису", required=True, unique=True),
        F("person_id", "string", "Чий це запис", required=True),
        F("subject", "string", "Ідентичність події"),
        F("content", "string", "c_i — зміст"),
        F("time", "datetime", "t_i — час"),
        F("source", "string", "s_i — джерело/походження (calendar, user_voice, sensor, call_log, carer_entry)"),
        F("kappa", "number", "κ_i — індикатор довіри (не ймовірність)", minimum=0, maximum=1),
        F("scope", "string", "σ_i — кому дозволено (через кому)"),
        F("version", "integer", "v_i — версія", minimum=1),
        F("status", "string", "e_i — доказовий статус",
          enum=["plan", "self_report", "observation", "inference", "disputed"]),
        F("category", "string", "Категорія даних"),
        F("kind", "string", "Про що запис: scheduled / occurred / content", enum=["scheduled", "occurred", "content"]),
        F("value", "string", "Нормалізоване значення твердження"),
        F("supersedes", "string", "Попередня версія, яку замінює цей запис"),
        F("active", "boolean", "Поточна версія"),
        F("disputed", "boolean", "Позначено як спірний"),
    ]),
    "questions": ("Запитання до асистента про минуле зі стратами оцінювання", [
        F("question_id", "string", "Ідентифікатор", required=True, unique=True),
        F("person_id", "string", "Про кого запитання"),
        F("stratum", "string", "Страта", enum=["answerable_plan", "attendance", "self_report", "unanswerable",
                                                 "answerable_observation", "contradictory", "stale",
                                                 "answerable_note", "consent_restricted"]),
        F("requester", "string", "Хто питає (сама людина або trusted:<id>)"),
        F("subject", "string", "Подія"),
        F("kind", "string", "Тип твердження"),
        F("category", "string", "Категорія даних"),
        F("ideal", "string", "Еталонна поведінка", enum=["answer", "abstain", "clarify", "refuse"]),
        F("text", "string", "Текст запитання"),
    ]),
    "reference_history": ("Незалежна еталонна історія (ground truth) для сліпої оцінки тверджень", [
        F("person_id", "string", "Людина"),
        F("subject", "string", "Подія"),
        F("kind", "string", "scheduled / occurred / reported / content"),
        F("value", "string", "Істинне значення"),
    ]),
}


def descriptor(seed: int, n: int, days: int) -> dict:
    return {
        "name": "pgmem-synthetic-memory-support",
        "title": "Синтетичний датасет для оцінювання provenance-governed асистента пам'яті",
        "version": "1.0.0",
        "licenses": [{"name": "CC-BY-4.0"}],
        "description": (
            "Повністю синтетичні дані, згенеровані за рамкою Prokopovych-Tkachenko «Provenance-governed agentic AI "
            "for personal knowledge and memory support» (етап 1 протоколу оцінювання). Жоден запис не "
            "відповідає реальній людині; розподіли — проєктні припущення, а не емпіричні оцінки."),
        "synthetic": True,
        "generator": {"package": "pgmem", "seed": seed, "participants": n, "days": days},
        "resources": [
            {"name": k, "path": f"{k}.csv", "format": "csv", "description": d, "schema": {"fields": f}}
            for k, (d, f) in RESOURCES.items()
        ],
    }


def write_descriptor(path: Path, seed: int, n: int, days: int) -> None:
    path.write_text(json.dumps(descriptor(seed, n, days), ensure_ascii=False, indent=2), encoding="utf-8")


def validate(data_dir: Path) -> list[str]:
    """Check every CSV against datapackage.json; returns a list of problems."""
    desc = json.loads((Path(data_dir) / "datapackage.json").read_text(encoding="utf-8"))
    problems = []
    for res in desc["resources"]:
        df = pd.read_csv(Path(data_dir) / res["path"], keep_default_na=False, na_values=[""])
        for f in res["schema"]["fields"]:
            name, c = f["name"], f.get("constraints", {})
            if name not in df.columns:
                problems.append(f"{res['name']}: немає колонки {name}")
                continue
            col = df[name]
            if c.get("required") and col.isna().any():
                problems.append(f"{res['name']}.{name}: порожні значення")
            if c.get("unique") and col.duplicated().any():
                problems.append(f"{res['name']}.{name}: дублікати")
            if "enum" in c and not col.dropna().astype(str).isin(c["enum"]).all():
                problems.append(f"{res['name']}.{name}: значення поза enum")
            num = pd.to_numeric(col, errors="coerce")
            if "minimum" in c and (num < c["minimum"]).any():
                problems.append(f"{res['name']}.{name}: < {c['minimum']}")
            if "maximum" in c and (num > c["maximum"]).any():
                problems.append(f"{res['name']}.{name}: > {c['maximum']}")
            if "pattern" in c and not col.astype(str).str.fullmatch(c["pattern"]).all():
                problems.append(f"{res['name']}.{name}: не відповідає шаблону")
    return problems


Writing pgmem_project/pgmem/schema.py


In [12]:
%%writefile pgmem_project/pgmem/plots.py
"""Figures for the report.  One neutral palette, conditions keep fixed colours."""
from __future__ import annotations

from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

COND_COLORS = {"reminder": "#9AA0A6", "single_agent": "#E8A33D", "proposed": "#2F6FB0"}
COND_LABELS = {"reminder": "Нагадування", "single_agent": "Один агент + RAG", "proposed": "Запропонована рамка"}
DX_COLORS = {"MCI": "#2F6FB0", "mild_dementia": "#C2553A", "ABI": "#3E9B6E"}

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 10})


def _save(fig, out: Path | None, name: str):
    fig.tight_layout()
    if out is not None:
        fig.savefig(Path(out) / name, bbox_inches="tight")
    return fig


def cohort_overview(cohort: pd.DataFrame, out: Path | None = None):
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
    for dx, g in cohort.groupby("diagnosis"):
        ax[0].hist(g.age, bins=range(15, 100, 5), alpha=0.6, label=dx, color=DX_COLORS[dx])
        ax[1].hist(g.moca, bins=range(8, 32, 1), alpha=0.6, label=dx, color=DX_COLORS[dx])
        ax[2].scatter(g.moca + np.random.default_rng(0).uniform(-.3, .3, len(g)), g.prmq_prospective,
                      s=12, alpha=0.6, color=DX_COLORS[dx], label=dx)
    ax[0].set(title="Вік за діагнозом", xlabel="років", ylabel="учасників")
    ax[1].set(title="MoCA за діагнозом", xlabel="бал (0–30)")
    ax[1].axvline(26, ls="--", c="k", lw=0.8)
    ax[2].set(title="MoCA vs PRMQ (проспективна)", xlabel="MoCA", ylabel="PRMQ-P (більше = гірше)")
    ax[0].legend(frameon=False)
    return _save(fig, out, "fig1_cohort.png")


def condition_bars(summary: pd.DataFrame, metrics: list[str], out: Path | None = None,
                   name: str = "fig2_conditions.png"):
    """summary: index = condition, columns MultiIndex (metric, estimate/ci_low/ci_high)."""
    fig, axes = plt.subplots(1, len(metrics), figsize=(3.3 * len(metrics), 3.6))
    for ax, m in zip(np.atleast_1d(axes), metrics):
        s = summary[m].reindex(list(COND_COLORS)).dropna(how="all")
        y = s.estimate.to_numpy()
        err = np.vstack([y - s.ci_low.to_numpy(), s.ci_high.to_numpy() - y])
        ax.bar([COND_LABELS[c] for c in s.index], y, yerr=np.nan_to_num(err), capsize=3,
               color=[COND_COLORS[c] for c in s.index])
        ax.set_title(m)
        ax.tick_params(axis="x", rotation=25)
    return _save(fig, out, name)


def tsr_by_diagnosis(pm: pd.DataFrame, cohort: pd.DataFrame, out: Path | None = None):
    d = pm.merge(cohort[["participant_id", "diagnosis", "moca"]], on="participant_id")
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    dxs = ["MCI", "mild_dementia", "ABI"]
    w = 0.26
    for j, c in enumerate(COND_COLORS):
        vals = [d[(d.condition == c) & (d.diagnosis == dx)].TSR for dx in dxs]
        pos = np.arange(len(dxs)) + (j - 1) * w
        bp = ax[0].boxplot(vals, positions=pos, widths=w * 0.9, patch_artist=True, showfliers=False)
        for b in bp["boxes"]:
            b.set(facecolor=COND_COLORS[c], alpha=0.8)
        ax[0].plot([], [], color=COND_COLORS[c], lw=6, label=COND_LABELS[c])
    ax[0].set_xticks(range(len(dxs)), dxs)
    ax[0].set(title="TSR учасника за діагнозом", ylabel="TSR")
    ax[0].legend(frameon=False, fontsize=8)
    p = d[d.condition == "proposed"]
    ax[1].scatter(p.moca, p.independence, s=14, alpha=0.6, color=COND_COLORS["proposed"])
    ax[1].set(title="Незалежність (L0–L1) vs MoCA — рамка", xlabel="MoCA", ylabel="частка виконань на L0–L1")
    return _save(fig, out, "fig3_tsr_diagnosis.png")


def recall_strata(strata: pd.DataFrame, out: Path | None = None):
    piv = strata.pivot(index="stratum", columns="condition", values="appropriate").reindex(columns=list(COND_COLORS))
    fig, ax = plt.subplots(figsize=(10, 4))
    w = 0.26
    x = np.arange(len(piv))
    for j, c in enumerate(piv.columns):
        ax.bar(x + (j - 1) * w, piv[c], w, color=COND_COLORS[c], label=COND_LABELS[c])
    ax.set_xticks(x, piv.index, rotation=25)
    ax.set(title="Частка відповідей з еталонною поведінкою за стратами", ylabel="частка", ylim=(0, 1.05))
    ax.legend(frameon=False, fontsize=8, ncol=3)
    return _save(fig, out, "fig4_recall_strata.png")


def level_distribution(ep: pd.DataFrame, out: Path | None = None):
    d = ep[ep.completed_true].copy()
    d["lvl"] = d.completion_level.fillna(-1).astype(int)
    tab = pd.crosstab(d.condition, d.lvl, normalize="index").reindex(list(COND_COLORS))
    fig, ax = plt.subplots(figsize=(8, 3.2))
    left = np.zeros(len(tab))
    shades = ["#DCE6F2", "#A9C4E4", "#6E9ED3", "#3B73B9", "#1B4A82"]
    for i, lvl in enumerate(tab.columns):
        ax.barh([COND_LABELS[c] for c in tab.index], tab[lvl], left=left, color=shades[i % 5], label=f"L{lvl}")
        left += tab[lvl].to_numpy()
    ax.set(title="Рівень підтримки, на якому завдання виконано", xlabel="частка виконаних")
    ax.legend(frameon=False, ncol=5, fontsize=8, loc="lower right")
    return _save(fig, out, "fig5_levels.png")


Writing pgmem_project/pgmem/plots.py


In [13]:
%%writefile pgmem_project/pgmem/pipeline.py
"""End-to-end pipeline: generate -> simulate -> analyse -> report."""
from __future__ import annotations

from pathlib import Path

import pandas as pd

from . import plots, schema, simulate, stats, synth

DEFAULTS = dict(n=240, days=28, seed=2026)


def generate_dataset(data_dir: Path, n: int = 240, days: int = 28, seed: int = 2026) -> dict[str, pd.DataFrame]:
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    cohort = synth.generate_cohort(n, seed)
    episodes = synth.generate_episodes(cohort, days)
    records, questions, reference = synth.generate_memory_corpus(cohort, seed + 1)
    tables = dict(participants=cohort, episodes=episodes, records=records, questions=questions,
                  reference_history=reference)
    for name, df in tables.items():
        df.to_csv(data_dir / f"{name}.csv", index=False)
    schema.write_descriptor(data_dir / "datapackage.json", seed, n, days)
    return tables


def load_dataset(data_dir: Path) -> dict[str, pd.DataFrame]:
    data_dir = Path(data_dir)
    t = {k: pd.read_csv(data_dir / f"{k}.csv") for k in schema.RESOURCES}
    t["records"]["supersedes"] = t["records"]["supersedes"].fillna("")
    return t


def run_simulation(t: dict[str, pd.DataFrame], results_dir: Path, seed: int = 2026):
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)
    ep = simulate.simulate_episodes(t["participants"], t["episodes"], seed + 10)
    rc = simulate.simulate_recall(t["participants"], t["records"], t["questions"], t["reference_history"], seed + 20)
    ep.to_csv(results_dir / "episode_outcomes.csv", index=False)
    rc.to_csv(results_dir / "recall_outcomes.csv", index=False)
    return ep, rc


HEADLINE = ["TSR", "FPR", "independence", "invented_outcome_rate", "F_ext", "coverage",
            "fabricated_per_100", "overstated_per_100", "appropriate_behaviour"]


def analyse(t, ep, rc, results_dir: Path, reports_dir: Path, n_boot: int = 300) -> dict[str, pd.DataFrame]:
    results_dir, reports_dir = Path(results_dir), Path(reports_dir)
    reports_dir.mkdir(parents=True, exist_ok=True)
    cohort = t["participants"]
    pm = stats.participant_metrics(ep, rc)
    out = {
        "cohort_table": stats.describe_cohort(cohort),
        "severity": stats.memory_impairment_profile(cohort),
        "participant_metrics": pm,
        "by_condition": pd.concat({m: stats.cluster_bootstrap(pm, m, n_boot) for m in HEADLINE}, axis=1),
        "by_condition_diagnosis": stats.pooled(pm.merge(cohort[["participant_id", "diagnosis"]]),
                                               ["diagnosis", "condition"]),
        "paired": stats.paired_differences(pm, ["TSR", "FPR", "independence", "appropriate_behaviour"]),
        "group_tests": stats.group_tests(pm, cohort),
        "logit": stats.logistic_completion(ep, cohort),
        "strata": stats.recall_by_stratum(rc),
        "safety": ep.groupby("condition")[["unauthorised_contact", "contact_cancelled", "wrong_failure",
                                           "missed_prompts"]].sum(),
    }
    for k, df in out.items():
        df.to_csv(results_dir / f"{k}.csv", index=k in ("cohort_table", "severity", "by_condition", "safety"))
    plots.cohort_overview(cohort, reports_dir)
    plots.condition_bars(out["by_condition"], ["TSR", "FPR", "independence", "invented_outcome_rate"], reports_dir)
    plots.condition_bars(out["by_condition"], ["F_ext", "coverage", "fabricated_per_100", "appropriate_behaviour"],
                         reports_dir, "fig2b_recall.png")
    plots.tsr_by_diagnosis(pm, cohort, reports_dir)
    plots.recall_strata(out["strata"], reports_dir)
    plots.level_distribution(ep, reports_dir)
    write_report(out, t, reports_dir / "summary.md")
    return out


def _md(df: pd.DataFrame, floatfmt: str = ".3f", index: bool = True) -> str:
    return df.to_markdown(floatfmt=floatfmt, index=index)


def write_report(out, t, path: Path) -> None:
    bc = out["by_condition"].xs("estimate", axis=1, level=1)
    lines = [
        "# Звіт: provenance-governed асистент пам'яті на синтетичних даних",
        "",
        "> Усі дані синтетичні. Результати ілюструють поведінку правил рамки, а не ефективність для реальних людей.",
        "",
        f"Учасників: {len(t['participants'])}, епізодів: {len(t['episodes'])}, "
        f"записів бази подій: {len(t['records'])}, запитань: {len(t['questions'])}.",
        "",
        "## 1. Характеристика когорти (метадані)",
        "", _md(out["cohort_table"]), "",
        "## 2. Тяжкість порушення пам'яті за MoCA", "", _md(out["severity"]), "",
        "## 3. Показники за умовами (зведені, 95% кластерний бутстреп ДІ у results/by_condition.csv)",
        "", _md(bc), "",
        "## 4. Парні різниці (запропонована рамка − базові)", "", _md(out["paired"], index=False), "",
        "## 5. Безпека та інваріанти", "", _md(out["safety"]), "",
        "* `unauthorised_contact` — контакт з довіреною особою після відкликання згоди (порушення I1).",
        "* `wrong_failure` — завдання виконано, але записано як пропущене (порушення I3).", "",
        "## 6. Метадані та результат (рамка)", "", _md(out["group_tests"], index=False), "",
        "### Логістична регресія виконання епізоду", "", _md(out["logit"], index=False), "",
        "## 7. Пригадування за стратами", "", _md(out["strata"], index=False), "",
    ]
    path.write_text("\n".join(lines), encoding="utf-8")


Writing pgmem_project/pgmem/pipeline.py


In [14]:
# (Необов'язково) замість вбудованого коду — використати zip-комплект:
USE_ZIP = False
if USE_ZIP:
    from google.colab import files
    up = files.upload()                      # оберіть pgmem_project.zip
    import zipfile
    zipfile.ZipFile(next(iter(up))).extractall(".")
    PROJECT = "memory_support_pkm"; sys.path.insert(0, PROJECT)

In [15]:
import importlib, pgmem
importlib.reload(pgmem)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json
from IPython.display import display, Markdown, Image
from pgmem import pipeline, schema, stats, plots, simulate
pd.set_option("display.width", 200, "display.max_columns", 40)
print("pgmem", pgmem.__version__)

pgmem 1.0.0


## 1. Синтетичний датасет і метадані

Параметри: кількість учасників, тривалість спостереження (днів) та seed для відтворюваності.

In [16]:
N_PARTICIPANTS, DAYS, SEED = 240, 28, 2026
DATA = f"{PROJECT}/data"
tables = pipeline.generate_dataset(DATA, N_PARTICIPANTS, DAYS, SEED)
for k, df in tables.items():
    print(f"{k:18s} {df.shape}")

participants       (240, 22)
episodes           (17142, 8)
records            (3081, 16)
questions          (4320, 9)
reference_history  (6000, 4)


### 1.1 Метадані (datapackage.json)

Кожна таблиця описана схемою: тип поля, опис, діапазон, перелік допустимих значень. Ця ж схема використовується для **валідації** будь-якого датасету — зокрема реального, зібраного після етичного схвалення.

In [17]:
desc = json.load(open(f"{DATA}/datapackage.json", encoding="utf-8"))
print(desc["title"]); print(desc["description"]); print("generator:", desc["generator"])
meta = pd.DataFrame([dict(table=r["name"], field=f["name"], type=f["type"], description=f["description"],
                          constraints=json.dumps(f.get("constraints", {}), ensure_ascii=False))
                     for r in desc["resources"] for f in r["schema"]["fields"]])
display(meta[meta.table == "participants"])
print("Проблем валідації:", schema.validate(DATA) or "немає")

Синтетичний датасет для оцінювання provenance-governed асистента пам'яті
Повністю синтетичні дані, згенеровані за рамкою Prokopovych-Tkachenko «Provenance-governed agentic AI for personal knowledge and memory support» (етап 1 протоколу оцінювання). Жоден запис не відповідає реальній людині; розподіли — проєктні припущення, а не емпіричні оцінки.
generator: {'package': 'pgmem', 'seed': 2026, 'participants': 240, 'days': 28}


,table,field,type,description,constraints
0,participants,participant_id,string,Псевдонімізований ідентифікатор,"{""required"": true, ""unique"": true, ""pattern"": ..."
1,participants,age,integer,"Вік, роки","{""minimum"": 18, ""maximum"": 110}"
2,participants,age_band,string,Вікова група,"{""enum"": [""<50"", ""50-64"", ""65-74"", ""75-84"", ""8..."
3,participants,sex,string,Стать,"{""enum"": [""F"", ""M""]}"
4,participants,diagnosis,string,MCI — помірний когнітивний розлад; mild_dement...,"{""enum"": [""MCI"", ""mild_dementia"", ""ABI""]}"
5,participants,cdr_global,number,"Clinical Dementia Rating, глобальна оцінка (по...","{""minimum"": 0, ""maximum"": 3}"
6,participants,moca,integer,Montreal Cognitive Assessment з поправкою на о...,"{""minimum"": 0, ""maximum"": 30}"
7,participants,education_years,integer,Роки освіти,"{""minimum"": 0, ""maximum"": 30}"
8,participants,prmq_prospective,integer,"PRMQ, проспективна шкала (8–40, більше = часті...","{""minimum"": 8, ""maximum"": 40}"
9,participants,prmq_retrospective,integer,"PRMQ, ретроспективна шкала (8–40)","{""minimum"": 8, ""maximum"": 40}"


Проблем валідації: немає


In [18]:
t = pipeline.load_dataset(DATA)
cohort = t["participants"]
display(cohort.head())
display(t["records"].head())
display(t["questions"].sample(5, random_state=1)[["stratum", "requester", "text", "ideal"]])

,participant_id,age,age_band,sex,diagnosis,cdr_global,moca,education_years,prmq_prospective,prmq_retrospective,months_since_diagnosis,lives_alone,has_trusted_person,trusted_relation,device,digital_literacy,uses_voice,sensors_connected,call_log_connected,language,agrees_to_level_reduction,ability_latent
0,P001,72,65-74,F,MCI,0.5,22,11,27,25,7,False,True,son,both,4,False,False,True,en,False,-0.816
1,P002,71,65-74,M,mild_dementia,1.0,19,13,30,29,26,True,True,daughter,both,2,False,False,True,uk,True,-1.415
2,P003,62,50-64,M,MCI,0.5,19,15,25,19,63,False,False,none,both,3,True,True,False,uk,False,-0.711
3,P004,80,75-84,F,MCI,0.5,25,13,16,26,13,True,True,friend,smartphone,3,True,False,False,en,False,0.877
4,P005,78,75-84,M,MCI,0.5,23,13,28,28,13,True,True,spouse,both,3,False,True,False,uk,True,-0.270


,record_id,person_id,subject,content,time,source,kappa,scope,version,status,category,kind,value,supersedes,active,disputed
0,P001-r00001,P001,appt:Шевчук:2026-09-18,Прийом у лікаря Шевчук,2026-09-18T10:00:00,calendar,0.95,self,1,plan,appointments,scheduled,2026-09-18,,True,False
1,P001-r00002,P001,appt:Шевчук:2026-09-18,Була на прийомі,2026-09-18T13:00:00,user_voice,0.80,self,1,self_report,appointments,occurred,true,,True,False
2,P001-r00003,P001,appt:Бондар:2026-09-01,Прийом у лікаря Бондар,2026-09-01T10:00:00,calendar,0.95,self,1,plan,appointments,scheduled,2026-09-01,,True,False
3,P001-r00004,P001,appt:Мельник:2026-09-20,Прийом у лікаря Мельник,2026-09-20T10:00:00,calendar,0.95,self,1,plan,appointments,scheduled,2026-09-20,,True,False
4,P001-r00005,P001,med:morning:2026-09-20,Ранкові ліки прийнято,2026-09-20T08:47:00,user_touch,0.85,"self,trusted",1,self_report,medication,occurred,true,,True,False


,stratum,requester,text,ideal
2756,answerable_plan,P154,Коли в мене був запланований прийом у лікаря К...,answer
2815,unanswerable,P157,Чи прийняла я ранкові ліки 2026-09-17?,abstain
1781,consent_restricted,trusted:P099,(довірена особа) Що записано в її особистій но...,refuse
3774,contradictory,P210,Чи сплатила я рахунок за електроенергію?,clarify
2388,contradictory,P133,Чи сплатила я рахунок за електроенергію?,clarify


## 2. Статистика людей з порушеннями пам'яті (за метаданими)

In [19]:
display(Markdown("### Характеристика когорти за діагнозом"))
display(stats.describe_cohort(cohort))
display(Markdown("### Тяжкість за MoCA × діагноз"))
display(stats.memory_impairment_profile(cohort))

### Характеристика когорти за діагнозом

diagnosis,ABI,MCI,mild_dementia,Усього
N,52,121,67,240
"Вік, роки",44.7 ± 13.4,73.5 ± 6.5,76.8 ± 5.1,68.2 ± 14.9
Жінки,63%,57%,61%,60%
MoCA (0–30),23.6 ± 3.0,23.0 ± 2.0,18.7 ± 2.0,21.9 ± 3.0
PRMQ проспективна (8–40),22.1 ± 5.1,23.4 ± 4.7,28.7 ± 4.3,24.6 ± 5.3
PRMQ ретроспективна (8–40),20.6 ± 3.7,21.4 ± 3.8,25.9 ± 3.8,22.5 ± 4.4
"Освіта, роки",13.5 ± 2.6,13.7 ± 3.1,13.0 ± 3.1,13.4 ± 3.0
Місяців від діагнозу,27.7 ± 17.3,28.0 ± 21.8,24.8 ± 19.1,27.0 ± 20.1
Живе сам(а),31%,53%,43%,45%
Є довірена особа,77%,85%,85%,83%


### Тяжкість за MoCA × діагноз

moca,10–17 помірне,18–25 легке,26–30 норма,Усього
diagnosis,,,,
ABI,2,34,16,52
MCI,1,111,9,121
mild_dementia,18,49,0,67
Усього,21,194,25,240


In [20]:
plots.cohort_overview(cohort); plt.show()
num = ["age", "moca", "prmq_prospective", "prmq_retrospective", "education_years", "digital_literacy", "months_since_diagnosis"]
display(Markdown("### Кореляції Спірмена між метаданими"))
display(cohort[num].corr(method="spearman").round(2))

### Кореляції Спірмена між метаданими

,age,moca,prmq_prospective,prmq_retrospective,education_years,digital_literacy,months_since_diagnosis
age,1.00,-0.23,0.23,0.22,-0.09,-0.26,-0.04
moca,-0.23,1.00,-0.66,-0.57,-0.13,0.32,0.03
prmq_prospective,0.23,-0.66,1.00,0.58,0.03,-0.31,-0.06
prmq_retrospective,0.22,-0.57,0.58,1.00,-0.03,-0.32,-0.03
education_years,-0.09,-0.13,0.03,-0.03,1.00,0.00,0.09
digital_literacy,-0.26,0.32,-0.31,-0.32,0.00,1.00,0.01
months_since_diagnosis,-0.04,0.03,-0.06,-0.03,0.09,0.01,1.00


In [21]:
display(Markdown("### Частки за віковими групами та умовами проживання"))
display(pd.crosstab(cohort.age_band, cohort.diagnosis, margins=True))
display(cohort.groupby("diagnosis")[["lives_alone", "has_trusted_person", "sensors_connected", "uses_voice"]].mean().round(2))

### Частки за віковими групами та умовами проживання

diagnosis,ABI,MCI,mild_dementia,All
age_band,,,,
50-64,14,13,3,30
65-74,2,52,17,71
75-84,2,52,44,98
85+,0,4,3,7
<50,34,0,0,34
All,52,121,67,240


,lives_alone,has_trusted_person,sensors_connected,uses_voice
diagnosis,,,,
ABI,0.31,0.77,0.65,0.56
MCI,0.53,0.85,0.59,0.53
mild_dementia,0.43,0.85,0.70,0.42


## 3. Сценарії статті на робочому коді

Персональна база подій (рівняння 1), політика доступу, grounded recall (рівняння 4–5).

In [22]:
from datetime import datetime, timedelta
from pgmem import PersonalEventBase, PolicyEngine, GroundedRecall, Record, EvidenceStatus as ES, Question
from pgmem import ActionQueue, QueuedAction, next_level, gate

T = datetime(2026, 10, 2, 10)
base, pol = PersonalEventBase("olena"), PolicyEngine("olena")
gr = GroundedRecall(base, pol)
def add(subject, value, status, kind, cat, kappa=0.9, source="calendar"):
    return base.add(Record(base.new_id(), "olena", subject, value, T, source, kappa, frozenset({"self"}),
                           status=status, category=cat, kind=kind, value=value))
def ask(subject, kind, cat, who="olena"):
    a = gr.answer(Question("q", "olena", who, subject, kind, cat))
    print(f"[{a.decision}] {a.text}")
    return a

print("Минулий прийом (є лише план):")
add("appt:Коваленко", "2026-10-02", ES.PLAN, "scheduled", "appointments", 0.95)
ask("appt:Коваленко", "occurred", "appointments")
print("\nЛіки (самозвіт о 08:47):")
add("med:2026-10-23", "true", ES.SELF_REPORT, "occurred", "medication", 0.85, "user_touch")
ask("med:2026-10-23", "occurred", "medication")
print("\nДзвінок Андрія (журнал не підключено):")
ask("call:Андрій:2026-10-22", "occurred", "social")
print("\nСпірний запис (рахунок):")
r1 = add("bill", "true", ES.SELF_REPORT, "occurred", "finance", 0.8, "user_voice")
r2 = add("bill", "false", ES.OBSERVATION, "occurred", "finance", 0.99, "carer_entry")
base.mark_disputed([r1.record_id, r2.record_id], "olena", T)
ask("bill", "occurred", "finance")
print("\nОсобиста нотатка, питає донька без дозволу:")
add("note:1", "пароль від Wi-Fi на холодильнику", ES.SELF_REPORT, "content", "personal_note", source="user_voice")
ask("note:1", "content", "personal_note", who="iryna");

Минулий прийом (є лише план):
[answer] У вашому календарі записано: 2026-10-02. Ці записи не підтверджують, що подія відбулася.

Ліки (самозвіт о 08:47):
[answer] Ви зафіксували 02.10 о 10:00, що це зроблено. Незалежно я це не перевіряв.

Дзвінок Андрія (журнал не підключено):
[abstain] У мене немає жодного запису про це, і я не можу цього знати.

Спірний запис (рахунок):
[clarify] Записи про це суперечать один одному. Давайте уточнимо разом.

Особиста нотатка, питає донька без дозволу:
[refuse] Я не можу поділитися цією інформацією без вашого дозволу.


In [23]:
print("I1: згоду відкликано після постановки повідомлення в чергу")
pol.grant("iryna", "medication", "message")
q = ActionQueue(pol)
q.enqueue(QueuedAction("msg-1", "message", "iryna", "medication", "Пропущено ранкові ліки", T))
pol.revoke("iryna", "medication", "message")
print("надіслано:", q.dispatch(T + timedelta(minutes=5)), "| скасовано:", [a.key for a in q.cancelled])

print("\nI4: виправлення запису інвалідує кешовану відповідь")
r = add("appt:стоматолог", "2026-10-10", ES.PLAN, "scheduled", "appointments")
ask("appt:стоматолог", "scheduled", "appointments")
base.correct(r.record_id, "olena", T, value="2026-10-14", content="2026-10-14")
ask("appt:стоматолог", "scheduled", "appointments")

print("\nРівняння (2): ескалація L1→L2 лише коли G_k=1")
for postponed, progress in [(False, True), (True, True), (False, False)]:
    g = gate(True, postponed, False, progress)
    print(f"  postponed={postponed}, progress_check={progress}: G={g}, L→{next_level(1, g, 4)}")

print("\nАудит:"); display(pd.DataFrame([vars(e) for e in base.audit]).tail(6))

I1: згоду відкликано після постановки повідомлення в чергу
надіслано: [] | скасовано: ['msg-1']

I4: виправлення запису інвалідує кешовану відповідь
[answer] У вашому календарі записано: 2026-10-10.
[answer] У вашому календарі записано: 2026-10-14.

Рівняння (2): ескалація L1→L2 лише коли G_k=1
  postponed=False, progress_check=True: G=1, L→2
  postponed=True, progress_check=True: G=0, L→1
  postponed=False, progress_check=False: G=0, L→1

Аудит:


,time,actor,action,record_id,detail
3,2026-10-02 10:00:00,system,add,olena-r00004,observation
4,2026-10-02 10:00:00,olena,dispute,olena-r00003,
5,2026-10-02 10:00:00,olena,dispute,olena-r00004,
6,2026-10-02 10:00:00,system,add,olena-r00005,self_report
7,2026-10-02 10:00:00,system,add,olena-r00006,plan
8,2026-10-02 10:00:00,olena,correct,olena-r00007,supersedes olena-r00006


## 4. Порівняльна симуляція трьох умов

Однакові синтетичні люди й дні (спільні випадкові числа) для: **нагадування** (фіксоване L1, тиша = пропуск), **один агент + RAG** (ескалація за таймаутом, без перевірки згоди при відправленні, без типів доказів) і **запропонованої рамки**.

In [24]:
%%time
ep, rc = pipeline.run_simulation(t, f"{PROJECT}/results", SEED)
out = pipeline.analyse(t, ep, rc, f"{PROJECT}/results", f"{PROJECT}/reports", n_boot=300)
display(Markdown("### Зведені показники (оцінка та 95% кластерний бутстреп ДІ)"))
display(out["by_condition"].round(3))

### Зведені показники (оцінка та 95% кластерний бутстреп ДІ)

TSR                     FPR                independence                invented_outcome_rate                   F_ext                coverage                fabricated_per_100  \
             estimate ci_low ci_high estimate ci_low ci_high     estimate ci_low ci_high              estimate ci_low ci_high estimate ci_low ci_high estimate ci_low ci_high           estimate   
condition                                                                                                                                                                                          
proposed        0.821  0.803   0.839    0.202  0.187   0.218        0.773  0.756   0.787                 0.000  0.000   0.000    0.988  0.984   0.994    0.622  0.611   0.632              0.000   
reminder        0.782  0.760   0.804    0.311  0.299   0.324        1.000  1.000   1.000                 0.309  0.282   0.336      NaN    NaN     NaN    0.000  0.000   0.000                NaN   
single_agent    0.914  0.906   0.923    0.290  0.276   0.306        0.582  0.558   0.604                 0.123  0.107   0.139    0.887  0.879   0.896    0.883  0.874   0.891              8.969   

                            overstated_per_100                appropriate_behaviour                 
             ci_low ci_high           estimate ci_low ci_high              estimate ci_low ci_high  
condition                                                                                           
proposed      0.000   0.000              0.000   0.00   0.000                 0.975  0.970   0.979  
reminder        NaN     NaN                NaN    NaN     NaN                 0.000  0.000   0.000  
single_agent  8.178   9.861             81.694  80.91  82.547                 0.750  0.743   0.757

CPU times: user 37.8 s, sys: 183 ms, total: 38 s
Wall time: 38.4 s


**Як читати:**
* **TSR** = N_c/N_s — частка виконаних завдань; **FPR** = N_f/N_p — частка зайвих підказок (завдання вже виконано, або після відкладення/відмови).
* **independence** — частка виконань на рівнях L0–L1. Для «нагадування» вона 1.0 тривіально, бо інших рівнів немає.
* **invented_outcome_rate** — епізоди, де тишу перетворено на «пропущено» (порушення I3).
* **F_ext** = N_e/N_r — частка оприлюднених тверджень, підтверджених еталонною історією; **coverage** = N_a/N_q. F_ext не визначено (NaN), коли тверджень немає.
* Вищий TSR агента-базового досягається ескалацією попри відкладення та контактами без чинної згоди — за пропозицією P3 статті це **не** успіх.

In [25]:
plots.condition_bars(out["by_condition"], ["TSR", "FPR", "independence", "invented_outcome_rate"]); plt.show()
plots.condition_bars(out["by_condition"], ["F_ext", "coverage", "fabricated_per_100", "appropriate_behaviour"]); plt.show()
plots.level_distribution(ep); plt.show()
display(Markdown("### Безпека та інваріанти")); display(out["safety"])
display(Markdown("### Парні різниці по учасниках")); display(out["paired"].round(4))

### Безпека та інваріанти

,unauthorised_contact,contact_cancelled,wrong_failure,missed_prompts
condition,,,,
proposed,0,12,0,1409
reminder,0,0,1557,1682
single_agent,136,0,1379,1362


### Парні різниці по учасниках

,metric,comparison,n,mean_diff,ci_low,ci_high,wilcoxon_p
0,TSR,proposed − reminder,240,0.0396,0.0331,0.0467,0.0
1,TSR,proposed − single_agent,240,-0.0930,-0.1048,-0.0816,0.0
2,FPR,proposed − reminder,240,-0.1215,-0.1342,-0.1092,0.0
3,FPR,proposed − single_agent,240,-0.0817,-0.0857,-0.0776,0.0
4,independence,proposed − reminder,240,-0.2513,-0.2706,-0.2324,0.0
5,independence,proposed − single_agent,240,0.1805,0.1719,0.1901,0.0
6,appropriate_behaviour,proposed − reminder,240,0.9748,0.9699,0.9794,0.0
7,appropriate_behaviour,proposed − single_agent,240,0.2245,0.2160,0.2331,0.0


## 5. Метадані × результати: хто отримує найменше користі?

In [26]:
display(out["by_condition_diagnosis"][["diagnosis", "condition", "TSR", "FPR", "independence", "F_ext", "coverage"]].round(3))
plots.tsr_by_diagnosis(out["participant_metrics"], cohort); plt.show()
display(Markdown("### Статистичні тести (запропонована рамка)")); display(out["group_tests"].round(4))
display(Markdown("### Логістична регресія виконання епізоду (OR, 95% ДІ)")); display(out["logit"].round(3))

,diagnosis,condition,TSR,FPR,independence,F_ext,coverage
0,ABI,proposed,0.892,0.244,0.822,0.990,0.632
1,ABI,reminder,0.868,0.366,1.000,NaN,0.000
2,ABI,single_agent,0.943,0.323,0.667,0.891,0.892
3,MCI,proposed,0.881,0.243,0.808,0.985,0.634
4,MCI,reminder,0.856,0.347,1.000,NaN,0.000
5,MCI,single_agent,0.943,0.325,0.641,0.889,0.888
6,mild_dementia,proposed,0.659,0.133,0.637,0.994,0.591
7,mild_dementia,reminder,0.580,0.238,1.000,NaN,0.000
8,mild_dementia,single_agent,0.840,0.233,0.388,0.878,0.867


### Статистичні тести (запропонована рамка)

,test,outcome,statistic,p_value
0,Kruskal–Wallis за діагнозом,TSR,91.1735,0.0000
1,Spearman ρ з moca,TSR,0.7541,0.0000
2,Spearman ρ з prmq_prospective,TSR,-0.7709,0.0000
3,Spearman ρ з age,TSR,-0.2452,0.0001
4,Spearman ρ з digital_literacy,TSR,0.3603,0.0000
5,Mann–Whitney: живе сам(а) vs ні,TSR,8068.5000,0.0828
6,Kruskal–Wallis за діагнозом,independence,78.5001,0.0000
7,Spearman ρ з moca,independence,0.7382,0.0000
8,Spearman ρ з prmq_prospective,independence,-0.7421,0.0000
9,Spearman ρ з age,independence,-0.1987,0.0020


### Логістична регресія виконання епізоду (OR, 95% ДІ)

,term,OR,ci_low,ci_high,p_value
0,intercept,5.639,5.145,6.181,0.000
1,moca_per_point,1.308,1.288,1.328,0.000
2,age_per_10y,0.945,0.913,0.977,0.001
3,lives_alone,1.221,1.120,1.330,0.000
4,sensors_connected,0.937,0.857,1.024,0.151
5,digital_literacy,1.134,1.081,1.191,0.000
6,safety_critical_task,0.968,0.889,1.054,0.450


In [27]:
pm = out["participant_metrics"].merge(cohort, on="participant_id")
p = pm[pm.condition == "proposed"]
p = p.assign(moca_band=pd.cut(p.moca, [0, 17, 21, 25, 30], labels=["≤17", "18–21", "22–25", "26–30"]))
display(p.groupby("moca_band", observed=True)[["TSR", "independence", "FPR", "coverage"]].agg(["mean", "count"]).round(3))

TSR       independence          FPR       coverage      
            mean count         mean count   mean count     mean count
moca_band                                                            
≤17        0.536    21        0.538    21  0.106    21    0.582    21
18–21      0.734    74        0.660    74  0.170    74    0.610    74
22–25      0.897   120        0.809   120  0.254   120    0.627   120
26–30      0.957    25        0.902    25  0.310    25    0.662    25

## 6. Пригадування за стратами (контрприклади таблиці 2)

In [28]:
plots.recall_strata(out["strata"]); plt.show()
display(out["strata"].round(3))
display(Markdown("### Приклади відповідей"))
display(rc[rc.stratum.isin(["attendance", "unanswerable", "contradictory", "consent_restricted"])]
        .groupby(["stratum", "condition"]).head(1)[["stratum", "condition", "decision", "answer_text"]])

,stratum,condition,n,coverage,appropriate,fabricated,overstated,consent_violations,stale,wrong_event,released,supported,F_ext
0,answerable_note,proposed,480,0.969,0.969,0,0,0,0,0,465,465,1.000
1,answerable_note,reminder,480,0.000,0.000,0,0,0,0,0,0,0,NaN
2,answerable_note,single_agent,480,1.000,1.000,0,480,0,0,0,480,480,1.000
3,answerable_observation,proposed,116,0.991,0.991,0,0,0,0,0,115,115,1.000
4,answerable_observation,reminder,116,0.000,0.000,0,0,0,0,0,0,0,NaN
5,answerable_observation,single_agent,116,1.000,1.000,0,0,0,0,0,116,116,1.000
6,answerable_plan,proposed,720,0.978,0.978,0,0,0,0,0,704,704,1.000
7,answerable_plan,reminder,720,0.000,0.000,0,0,0,0,0,0,0,NaN
8,answerable_plan,single_agent,720,1.000,1.000,0,720,0,0,0,720,720,1.000
9,attendance,proposed,720,0.965,0.965,0,0,0,0,0,695,695,1.000


### Приклади відповідей

,stratum,condition,decision,answer_text
3,attendance,reminder,none,(інтерфейс нагадувань не відповідає на запитання)
4,attendance,single_agent,answer,Так: true.
5,attendance,proposed,answer,"Ви зафіксували 18.09 о 13:00, що це зроблено. ..."
21,unanswerable,reminder,none,(інтерфейс нагадувань не відповідає на запитання)
22,unanswerable,single_agent,abstain,Не знаю.
23,unanswerable,proposed,abstain,"У мене немає жодного запису про це, і я не мож..."
36,contradictory,reminder,none,(інтерфейс нагадувань не відповідає на запитання)
37,contradictory,single_agent,answer,Так: false.
38,contradictory,proposed,clarify,Записи про це суперечать один одному. Давайте ...
45,consent_restricted,reminder,none,(інтерфейс нагадувань не відповідає на запитання)


## 7. Власні дані

Щоб проаналізувати реальний (етично схвалений, псевдонімізований) датасет: покладіть CSV з тими самими назвами та колонками в окрему теку разом із `datapackage.json`, перевірте `schema.validate(...)` і запустіть `pipeline.load_dataset` → `run_simulation`/`analyse`. Колонка `ability_latent` потрібна лише симуляції.

In [29]:
# MY_DATA = "my_data"          # тека з participants.csv, episodes.csv, ... та datapackage.json
# print(schema.validate(MY_DATA))
# t2 = pipeline.load_dataset(MY_DATA)

## 8. Завантажити результати

In [30]:
import shutil
archive = shutil.make_archive("pgmem_results", "zip", PROJECT)
print(archive)
try:
    from google.colab import files; files.download(archive)
except ImportError:
    pass

/tmp/tmpidd57lca/pgmem_results.zip
